# Llama-free + V31 + ATLAS20 + our Pool v5

Unscored experiment derived from the Llama-free notebook (reported LB 0.649), using V31 methods from the public 0.663 notebook. Those scores do not apply to this new combination.

## Attach on Kaggle

Copy and Edit the public **name-that-llm-llama-free-lora-ensemble** notebook to retain all its inputs. Import this notebook and add:

- **dhruvtibarewal/nlm-matched-clean**, containing `nlm_matched_clean.parquet`.
- **harrytsjan/qwen3-8b-poolv5-20261027**, containing our final checkpoint with `adapter/`, `readouts.npz`, `norms.safetensors`, and tokenizer.

Keep the source notebook's Qwen bases and public adapters, DeBERTa, complementary-stack inputs, spectral heads, public pool and ATLAS bundle. Use **2 x T4, internet off**. No Llama input is required.

Set `USE_OUR_POOLV5 = False` for the public Qwen3-8B control. True replaces that slot with our single final checkpoint (solo LB 0.602), weight 2.0 and twin 0.35; its Gaussian head is disabled. The saved heads, norms, label order and tokenization are preserved.

V31 additions: deduplicated and source-weighted external training; TF-IDF 0.25 and distortion 0.10; actual fitted family gates; ATLAS 20%; soft Sinkhorn 0.45 toward a 65% predicted / 35% train prior mixture; gated Bayesian typography. Spectral heads remain at 10%.


## Pipeline

Text and neural members, complementary stack, ATLAS20, spectral heads, soft balancing, and Bayesian typography. Source runtime safeguards remain enabled.


## Setup

This cell finds every input under `/kaggle/input` by what it is, not by a fixed path, so the notebook keeps working when a dataset gets a new version or mounts in a different folder.

- `model_dir` finds a base model from parts of its path (`/qwen-3/` and `/14b-base/`, and so on).
- `adapter_dir` and `folds(tag)` find dhruv's fold adapters: five fold adapters plus the full-data adapter for each tag.
- `checkpoint` finds Nishant's part-two checkpoints and returns their own config (layers kept, pooled layers, max length, end token), so they're read exactly the way they were trained.
- DeBERTa is copied to local disk first, because loading it straight from the dataset mount is a lot slower.

The block at the bottom is the commit preview. If the test file is the 3-row public one, it writes a placeholder `submission.parquet`, prints whether the stack's checkpoints are attached, and sets `PREVIEW` so the heavy cells are skipped.

In [ ]:
import glob, json, os, shutil

def model_dir(*parts):
    hits = sorted(os.path.dirname(c) for c in glob.glob('/kaggle/input/**/config.json', recursive=True) if all(p in c.lower() for p in parts))
    assert hits, parts
    return hits[0]

def adapter_dir(name):
    hits = glob.glob(f'/kaggle/input/**/{name}/heads.pt', recursive=True)
    assert hits, name
    return os.path.dirname(hits[0])

def checkpoint(name):
    meta = glob.glob(f'/kaggle/input/**/{name}/meta.json', recursive=True)
    assert meta, name
    return os.path.dirname(meta[0]), json.load(open(meta[0]))['cfg']

def local_copy(name):   # encoders load far faster from local disk than from the dataset mount
    src = next(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/meta.json', recursive=True) if f'/{name}/' in p or (name == 'mb_v5' and 'nlm-enc-v5' in p))
    shutil.copytree(src, f'/tmp/{name}', dirs_exist_ok=True)
    return f'/tmp/{name}'

DATA = next(p for p in ['/kaggle/input/competitions/name-that-llm', '/kaggle/input/name-that-llm'] if os.path.exists(p + '/train.csv'))
POOL = glob.glob('/kaggle/input/**/nlm_pool_v5.parquet', recursive=True)[0]
ATLAS = os.path.dirname(glob.glob('/kaggle/input/**/atlas_bundle.json', recursive=True)[0])

QWEN8, QWEN14 = model_dir('/qwen-3/', '/8b-base/'), model_dir('/qwen-3/', '/14b-base/')
QWEN25_14 = model_dir('/qwen2.5/', '/14b/')
folds = lambda tag: [adapter_dir(f'{tag}_f{k}') for k in range(5)] + [adapter_dir(tag)]
N14, N14_CFG = checkpoint('lora_qwen3_14b')
Q25, Q25_CFG = checkpoint('lora_qwen25_14b')
DEBERTA = local_copy('deberta_v3_large')

# ---- Fork addition: commit-run preview ------------------------------------------------------------------------------
# A commit run sees only the 3-row public test.csv. Every input above is resolved by now, so write a placeholder
# submission and skip the model cells (1.5-2 h of GPU for 3 rows); the hidden-test rerun runs the full pipeline.
import pandas as _pd
_test = _pd.read_csv(os.path.join(DATA, 'test.csv'))
PREVIEW = len(_test) <= 3 and set(_test['id']) <= {'00353fb1c541', '005f5a9a6139', '0062a53d5487'}
if PREVIEW:
    _pd.DataFrame({'id': _test['id'], 'model': 'gpt-5.5'}).to_parquet('/kaggle/working/submission.parquet', index=False)
    for _m in ['**/name-that-llm-train-lora24-s7/lora_out/job_final/COMPLETE.json',
               '**/ntl-qwen3-8b-lora-claude-head/**/final/COMPLETE.json',
               '**/ntl-train-lora-qwen25-instyle/lora_out/job_final/COMPLETE.json',
               '**/tokattn_debl23_final.pt', '**/ultrabert-380m-ntl', '**/ntl_extra_passages_claude.csv']:
        print('comp-stack input', _m, len(glob.glob('/kaggle/input/' + _m, recursive=True)), flush=True)
print('PREVIEW: placeholder submission, model cells skipped' if PREVIEW else f'FULL RUN on {len(_test)} test rows', flush=True)
USE_OUR_POOLV5 = True
EXTRA = glob.glob('/kaggle/input/**/nlm_matched_clean.parquet', recursive=True)
assert EXTRA, 'Attach dhruvtibarewal/nlm-matched-clean'
EXTRA = EXTRA[0]
if USE_OUR_POOLV5:
    from pathlib import Path
    import numpy as np
    import torch
    from safetensors.torch import load_file, save_file
    from transformers import AutoTokenizer
    roots = list(Path('/kaggle/input').rglob('qwen3-8b-poolv5-20261027'))
    assert len(roots) == 1, 'Attach harrytsjan/qwen3-8b-poolv5-20261027'
    candidates = [p.parent for p in roots[0].rglob('readouts.npz') if (p.parent/'adapter/adapter_model.safetensors').is_file() and (p.parent/'norms.safetensors').is_file()]
    finals = [p for p in candidates if p.name == 'job_final']
    candidates = finals or candidates
    assert len(candidates) == 1, candidates
    ck = candidates[0]
    OUR_POOLV5 = Path('/kaggle/working/our_poolv5_converted')
    OUR_POOLV5.mkdir(exist_ok=True)
    with np.load(ck/'readouts.npz', allow_pickle=False) as z:
        labels = z['classes'].astype(str).tolist()
        assert len(set(labels)) == 18
        assert z['pool_layers'].tolist() == [2,4,6,8,12,16,20,24]
        H = {}
        for dest, orig in [('plain','plain'),('centered','center')]:
            assert z[orig+'_coef'].shape == (18,32768)
            H[dest+'.weight'] = torch.tensor(z[orig+'_coef'].astype(np.float32))
            H[dest+'.bias'] = torch.tensor(z[orig+'_intercept'].astype(np.float32))
    for k,value in load_file(str(ck/'norms.safetensors')).items():
        H[k if k.startswith('norms.') else 'norms.'+k] = value.float().contiguous()
    save_file(H,str(OUR_POOLV5/'heads.safetensors'))
    (OUR_POOLV5/'meta.json').write_text(json.dumps({'labels':labels}))
    shutil.copytree(ck/'adapter',OUR_POOLV5/'adapter',dirs_exist_ok=True)
    OUR_TOKENIZER = str(ck/'tokenizer') if (ck/'tokenizer').is_dir() else QWEN8
    OUR_EOS = AutoTokenizer.from_pretrained(OUR_TOKENIZER).eos_token_id
    assert OUR_EOS is not None
    print('Our Pool v5:',ck,'EOS',OUR_EOS)
    del H


## Shared-prompt Gaussian head (`gauss.py`)

This is the idea in the notebook that's most worth understanding. Treat the features of a passage as

    x = (mean of its author) + (effect of its prompt) + noise

where the prompt effect is shared by every passage written for that prompt. It's fit on train, where the prompts are known: reduce the features with a PCA of the within-prompt residuals, then estimate the author means and two covariances, one for the prompt effect and one for the noise (both shrunk with Ledoit-Wolf). A joint diagonalisation makes every dimension independent, which keeps the maths cheap.

At test time:

- a passage on its own has its unknown prompt effect integrated out;
- a group of 2 or 3 passages that seem to share a prompt is scored jointly over every possible assignment of models, with the rule that no model answers the same prompt twice;
- bigger groups use a cavity EM approximation of the same thing.

The joint scoring is where the gain comes from. Once you know two passages share a prompt, whatever they have in common is the prompt, and whatever differs is the authors. The code is ported (Apache-2.0) from public notebooks by prvsiyan, kirilltym and sangam21. This file is only a library; the main pipeline imports it.

In [ ]:
%%writefile gauss.py
"""Shared-prompt Gaussian group head (ported, Apache-2.0, from public Kaggle notebooks by prvsiyan / kirilltym / sangam21).
Two-way model x = class mean + prompt effect + noise; exact (n<=3) or cavity-EM (n>3) injective group marginals."""
import itertools
import numpy as np
K = 18
DEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,
           "iters": 5, "repel": True, "sp_scale": 1.0}

# =================================================================== primitives ====
def _lse(A, axis=None, keepdims=False):
    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
    if keepdims: return out
    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

def log_softmax(L): return L - _lse(L, axis=1, keepdims=True)

def _lw_shrinkage(X):
    """Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered)."""
    from sklearn.covariance import ledoit_wolf_shrinkage
    return float(ledoit_wolf_shrinkage(X, assume_centered=True))

def _shrunk(S, a):
    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)

def _two_way(Z, y, pinv, G, iters=4):
    """additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds)."""
    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)
    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))
    for _ in range(iters):
        R = Z - P[pinv]
        for c in cls: mu[c] = R[y == c].mean(0)
        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]
    return mu, P, cnt_p

def fit(X, y, prompts, cfg=None):
    """fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.
    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller's cross-fit)."""
    c = dict(DEFAULT); c.update(cfg or {})
    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape
    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)
    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx
    if c["red"] == "none":
        V = np.eye(D)
    else:
        if c["red"] == "wpca":
            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]
        elif c["red"] == "pca": R = Xs
        else: raise ValueError(c["red"])
        d = int(min(c["d"], N - G - 1, D))
        # economy SVD through the (N x N) Gram matrix (N << D for the real views)
        if N < D:
            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]
            V = (R.T @ U) / np.sqrt(w)
        else:
            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T
    Z = Xs @ V; d = Z.shape[1]
    mu, P, ng = _two_way(Z, y, pinv, G)
    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)
    Se_raw = E.T @ E / dof
    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)
    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])
    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]
    # joint diagonalisation: A Se A' = I, A Sp A' = diag(lam)
    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)
    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li
    W = V @ A.T
    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,
            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}

def transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]

_MASK = {}
def _inj_mask(n, C):
    if (n, C) not in _MASK:
        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)
        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]
        _MASK[(n, C)] = bad
    return _MASK[(n, C)]

def singleton_ll(U, m, lam):
    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""
    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)

def joint_terms(U, m, lam):
    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""
    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]
    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)
    return a, B

def joint_table(a, B):
    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""
    n, C = a.shape; Q = np.zeros([C] * n)
    for i in range(n):
        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)
    for i, j in itertools.combinations(range(n), 2):
        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)
    return Q

def exact_marginals(Q, T=1.0, inj=True):
    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""
    n = Q.ndim; C = Q.shape[0]; Qt = Q / T
    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)
    lz = _lse(Qt); out = np.empty((n, C))
    for i in range(n):
        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz
    return out

def em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):
    """cavity EM for one group (any n >= 2): returns (n, C) log q."""
    n = len(U); v = lam / (1 + (n - 1) * lam)
    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))
    q = np.exp(log_softmax(L))
    for _ in range(iters):
        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v                  # cavity prompt-effect means (n, d)
        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T
        if repel:   # mean-field at-most-once: discourage classes the other members already hold
            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)
        q = np.exp(log_softmax(L))
    return log_softmax(L)

LAM_DIFF = 1e6
def lam_g(prm, cfg=None):
    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg 'diff'): then the joint likelihood
    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs
    centred prototypes) and the prompt term cancels exactly."""
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})
    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]

def group_logpost(U, prm, T1=None, Tg=None, cfg=None):
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)
    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg
    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)
    if n <= c["n_exact"]:
        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])
    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])

def scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):
    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""
    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))
    sizes = np.bincount(lab); s1 = sizes[lab] == 1
    T1 = prm["T1"] if T1 is None else T1
    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)
    for g in np.where(sizes >= 2)[0]:
        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)
    return out

## Complementary stack (`comp_stack.py`)

A second ensemble, built from the public notebooks of prvsiyan, Lucas Boesen, vnton37nguyn and kirilltym and ported by dhruv. It runs as its own process and writes one array of log-probabilities, which the main pipeline mixes in at 15%. Its members are deliberately different from the main ones: frozen features from several base models with simple heads on top, style n-grams, stylometry, token-attention heads and UltraBERT.

### 0. Configuration

Labels, grouping thresholds and blend weights. The weights are log-probability multipliers that came from an L-BFGS fit on prompt-grouped out-of-fold predictions in the original 0.563 notebook. A weight of 0 for a frozen model means its plain head isn't used at all, only its centred twin and Gaussian head.

The member lists say which model and which layers each member reads:

- `LM_HEADS`: frozen decoders (GLM-4-9B, Qwen3-8B, Qwen2.5-7B, Qwen3-4B), hidden states from two layers each, pooled either as a plain mean or as a "style" mean over stop-word and punctuation tokens only.
- `ENC_HEADS`: frozen encoders (DeBERTa-v3-large, ELECTRA-large, the desklib AI-text detector).
- `LORA_RUNS`: four LoRA classifiers trained by other people, used for inference only.
- `TOKATTN_HEADS`: a token-attention head on DeBERTa.

`GATE_LAM` and `BALANCE_ALPHA` are left over from the original notebook; nothing below reads them.

In [ ]:
%%writefile comp_stack.py
"""Complementary frozen-feature stack from the public notebooks of prvsiyan, Lucas Boesen, vnton37nguyn and Kirill Tymchyshyn
(Apache-2.0). Writes its raw log-probability blend for the main pipeline."""
# ============================================================ 0. configuration
import gc, glob, json, os, pathlib, re, time
import numpy as np, pandas as pd
try:
    import torch
except ImportError:   # local smoke without torch
    torch = None
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.sparse import csr_matrix, diags, hstack, vstack
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

SMOKE = os.environ.get('NTL_SMOKE') == '1'   # local code-path check: pseudo features instead of checkpoints, CPU
T0 = time.time()
LABELS = sorted(['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
                 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
                 'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
                 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'])
K = len(LABELS)
LAB2I = {l: i for i, l in enumerate(LABELS)}
FAMILY_IDX = {f: np.array([i for i, l in enumerate(LABELS) if l.startswith(f)]) for f in ('claude', 'gpt', 'gemini', 'glm')}

# ---- test-pool grouping
GROUP_THRESH, GROUP_LAYER, GROUP_MIN, GROUP_MAX = 0.60, 24, 2, 18   # mixed content + GLM-L24 distance, average linkage
STYLE_GROUP_THRESH = 0.90                                           # stylegram's own median-normalised content distance

# ---- blend weights: log-probability multipliers of the 0.563 notebook (its L-BFGS fit on prompt-grouped OOF)
W = {'stylegram': 1.516, 'stylometry_lgb': 0.063, 'ultrabert': 0.095,
     'q3_8b': 0.0, 'q25_7b': 0.0, 'q3_4b': 0.0, 'glm9b': 0.0, 'debv3l': 0.05, 'electl': 0.032, 'desk': 0.132,
     'lora24_public': 0.177, 'lora24_seed7': 0.113, 'lora24_claude': 0.177, 'lora20_q25_instyle': 0.17,
     'tokattn_q35': 0.153, 'tokattn_deb': 0.185}
TWIN_W = {'q3_8b': 1.5 * 0.219, 'q25_7b': 1.5 * 0.141, 'q3_4b': 1.5 * 0.083, 'glm9b': 1.5 * 0.110, 'debv3l': 0.25}
LORA_TWIN_W = {'lora24_public': 0.15, 'lora24_claude': 0.15}
GATE_LAM = {'claude_style': 1.5, 'claude_lora': 1.0, 'family': 0.25}
BALANCE_ALPHA = 0.5
EXTRA_MIN_LABELS, EXTRA_SHARE = 12, 0.5

# ---- the addition: shared-prompt Gaussian group heads on four measured views, weight = the centred twin of the same backbone,
#      reduction / temperatures from our production fit (research/gaussian/production_weights_v8g.json, 135 training prompts)
GAUSS_HEADS = {'gauss_q3_8b_L22': dict(src='q3_8b', part='L22_mean', d=384, T1=2.351, Tg=2.306, w=TWIN_W['q3_8b']),
               'gauss_glm9b_L8': dict(src='glm9b', part='L8_mean', d=384, T1=2.722, Tg=2.596, w=TWIN_W['glm9b']),
               'gauss_q3_4b_L4': dict(src='q3_4b', part='L4_mean', d=384, T1=2.725, Tg=2.578, w=TWIN_W['q3_4b']),
               'gauss_debv3l_L24': dict(src='debv3l', part='L24_mean', d=384, T1=3.112, Tg=3.069, w=TWIN_W['debv3l'])}
GAUSS_SETTING = dict(red='wpca', se_shrink='lw', sp_shrink='lw', lam_floor=1e-4, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0)

# ---- member specifications
LM_HEADS = [  # key, model_type, hidden size, parts of the plain head [(layer, pool)], extra parts extracted in the same pass, C
    ('glm9b', 'glm4', 4096, [(8, 'mean'), (3, 'mean')], [(GROUP_LAYER, 'mean')], 0.0025),   # first: its L24 builds the groups
    ('q3_8b', 'qwen3', 4096, [(7, 'mean'), (22, 'mean')], [], 0.0025),
    ('q25_7b', 'qwen2', 3584, [(6, 'mean'), (20, 'mean')], [], 0.0025),
    ('q3_4b', 'qwen3', 2560, [(4, 'style'), (7, 'style')], [(4, 'mean')], 0.0025)]
ENC_HEADS = [  # key, input directory name, parts, C
    ('debv3l', 'deberta-v3-large', [(24, 'mean'), (6, 'mean')], 0.005),
    ('electl', 'electra-large-discriminator', [(21, 'mean'), (24, 'style')], 0.005),
    ('desk', 'desklib-detector', [(23, 'mean'), (23, 'style')], 0.005)]
LORA_RUNS = [  # (trunk model_type, hidden, kept layers), pooled layers, [(key, checkpoint marker, norms file, anchor features file)]
    (('qwen3', 4096, 24), [2, 4, 6, 8, 12, 16, 20, 24], [
        ('lora24_public', '**/lora24/final/COMPLETE.json', 'heads.safetensors', 'anchor_features.npz'),
        ('lora24_seed7', '**/name-that-llm-train-lora24-s7/lora_out/job_final/COMPLETE.json', 'norms.safetensors', 'valid_features.npz'),
        ('lora24_claude', '**/ntl-qwen3-8b-lora-claude-head/**/final/COMPLETE.json', 'heads.safetensors', 'anchor_features.npz')]),
    (('qwen2', 3584, 20), [2, 4, 6, 8, 10, 12, 16, 20], [
        ('lora20_q25_instyle', '**/ntl-train-lora-qwen25-instyle/lora_out/job_final/COMPLETE.json', 'norms.safetensors', 'valid_features.npz')])]
LORA_MAX_LEN, LORA_BS = 447, 8
TOKATTN_HEADS = [('tokattn_deb', 'tokattn_debl23_final.pt', 'deberta')]



### Data and helpers

Loads train and test (or a small fake split in `SMOKE` mode, which is how the script can be tested on a laptop without GPUs). `LOGIT` is the running blend, and `add_member` adds `weight × log(probability)` to it, either for every row or only for the rows in recovered groups. `self_test_rows` takes the training passages of three known prompts so that the grouped code paths get exercised even in a 3-row run.

In [ ]:
%%writefile -a comp_stack.py
# ---- data
if SMOKE:
    _all = pd.read_csv(os.environ['NTL_DATA'] + '/train.csv')
    _p = _all['prompt_id'].drop_duplicates().sample(frac=1, random_state=0).tolist()
    train = _all[_all['prompt_id'].isin(_p[:30])].reset_index(drop=True)
    test = _all[_all['prompt_id'].isin(_p[30:42])].sample(frac=1, random_state=1).reset_index(drop=True)
    test = pd.concat([test[test['prompt_id'] == p].head(k) for p, k in zip(_p[30:42], [1, 1, 2, 2, 2, 3, 3, 4, 6, 9, 18, 18])]).reset_index(drop=True)
    if os.environ.get('NTL_TINY') == '1':
        test = test.head(3).reset_index(drop=True)
    SMOKE_Y = test['model'].map(LAB2I).to_numpy(dtype=int)
else:
    DATA_DIR = next(p for p in [pathlib.Path('/kaggle/input/competitions/name-that-llm'), pathlib.Path('/kaggle/input/name-that-llm')]
                    if (p / 'train.csv').exists())
    train, test = pd.read_csv(DATA_DIR / 'train.csv'), pd.read_csv(DATA_DIR / 'test.csv')
    if os.environ.get('NTL_FAKE_TEST'):   # timing probe: pool passages as a stand-in test (same rows/seed as infer.py --fake_test)
        test = pd.read_parquet(os.environ['NTL_POOL']).sample(int(os.environ['NTL_FAKE_TEST']), random_state=0)[['text']].reset_index(drop=True)
        test['id'] = [f'fake{i}' for i in range(len(test))]
test['text'] = test['text'].fillna('').astype(str)
TR, TE = train['text'].tolist(), test['text'].tolist()
y = train['model'].map(LAB2I).to_numpy(dtype=int)
PROMPT = train['prompt_id'].to_numpy(dtype=object)
TINY = len(TE) < 50   # the interactive 3-row test: exercise the group paths on training rows so the hidden rerun cannot hit untested code
DEV = 'cuda' if torch is not None and torch.cuda.is_available() else 'cpu'
print(f'train {train.shape}  test {test.shape}  gpus {torch.cuda.device_count() if torch is not None else 0}  smoke {SMOKE}', flush=True)

LOGIT = np.zeros((len(TE), K))           # the running log-probability blend


def add_member(key, prob, rows=None, weight=None):
    """Add weight * log(prob) to the blend (all rows, or the given row subset)."""
    w = W[key] if weight is None else weight
    lp = np.log(np.clip(prob, 1e-9, 1.0))
    if rows is None:
        LOGIT[:] += w * lp
    else:
        LOGIT[rows] += w * lp
    print(f'  + {key:20s} weight {w:.3f} on {len(TE) if rows is None else int(np.sum(rows))} rows  ({time.time() - T0:.0f}s)', flush=True)


def softmax_rows(S):
    S = S - S.max(1, keepdims=True)
    P = np.exp(S)
    return P / P.sum(1, keepdims=True)


def full_prob(P_part, classes):
    """Expand a (n, len(classes)) probability block to (n, K) on the global label order."""
    P = np.zeros((len(P_part), K))
    P[:, classes] = P_part
    return P


def group_centre_dense(X, groups):
    X = np.asarray(X, dtype=np.float32).copy()
    for g in np.unique(groups):
        i = groups == g
        X[i] -= X[i].mean(0)
    return X


def self_test_rows():
    """Training rows of the first three prompts, grouped by their true prompt (used only when the test file is tiny)."""
    idx = np.where(np.isin(PROMPT, pd.unique(PROMPT)[:3]))[0]
    return idx, np.unique(PROMPT[idx], return_inverse=True)[1]



### 1. Grouping the test passages by prompt

Two groupings, both with average linkage:

- `mixed_groups`: the mean of a content-word TF-IDF distance and an embedding distance from layer 24 of GLM-4-9B, cut at 0.60. The frozen models' centred twins and Gaussian heads use these groups.
- `normalised_content_groups`: the TF-IDF distance divided by its own median, cut at 0.90. The style n-gram twin uses this one.

Only groups of 2 to 18 passages count (`eligible`), since a prompt has at most 18 answers.

In [ ]:
%%writefile -a comp_stack.py
# ============================================================ 1. grouping the test pool
def average_linkage_groups(D, thresh):
    """Cluster ids (1-based) from a square distance matrix, average linkage, cut at thresh."""
    if len(D) < 2:
        return np.ones(len(D), dtype=int)
    D = np.asarray(D, dtype=np.float64).copy()
    np.fill_diagonal(D, 0.0)
    return fcluster(linkage(squareform(D, checks=False), method='average'), thresh, criterion='distance')


def content_cosine_distance(texts, max_features=None):
    X = TfidfVectorizer(stop_words='english', sublinear_tf=True, max_features=max_features, dtype=np.float32).fit_transform(texts)
    return np.clip(1.0 - (X @ X.T).toarray(), 0.0, 1.0)


def mixed_groups(texts, E, thresh=GROUP_THRESH):
    """Average of the content-word TF-IDF cosine distance and the cosine distance of mean-centred, L2-normalised embeddings E."""
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    Z = np.asarray(E, dtype=np.float64)
    Z = Z - Z.mean(0)
    Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12
    D = 0.5 * (content_cosine_distance(texts) + np.clip(1.0 - Z @ Z.T, 0.0, 2.0))
    return average_linkage_groups(D, thresh)


def normalised_content_groups(texts, thresh=STYLE_GROUP_THRESH):
    """The stylegram's grouping: content distance divided by its median off-diagonal value, cut at thresh."""
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    D = content_cosine_distance(texts, max_features=100_000)
    med = max(float(np.median(D[np.triu_indices(len(texts), 1)])), 1e-6)
    return average_linkage_groups(D / med, thresh)


def eligible(groups, lo=GROUP_MIN, hi=GROUP_MAX):
    size = np.bincount(groups)[groups]
    return (size >= lo) & (size <= hi)


def describe_groups(name, groups):
    size = np.bincount(groups)[groups]
    print(f'{name}: {len(groups)} rows | singletons {int((size == 1).sum())}  2-3: {int(((size >= 2) & (size <= 3)).sum())}'
          f'  4-18: {int(((size >= 4) & (size <= 18)).sum())}  >18: {int((size > 18).sum())}', flush=True)


G_TE, GROUP_ROWS = None, None   # set after the GLM pass (section 4)



### 2. Style n-grams, extra passages, family gates

Four TF-IDF views stacked into one sparse matrix: character 3-5 grams, word 1-3 grams, a "shape" view (every word turned into x/X, so only capitalisation, digits and punctuation are left) and a function-word view (content words replaced with `CONTENT`).

The training rows are the official train plus **in-style public passages**: generations from several public datasets that used a prompt bank in the host's format. Only prompts answered by at least 12 of the 18 models are kept, and together those rows get half of each class's weight, so they can't drown out the official rows.

Two linear SVMs come out of this: a plain one, and a centred one fit on prompt-centred rows, which replaces the plain prediction for the test passages that sit in a recovered group.

The family gates at the end (small SVMs inside Claude, GPT, Gemini and GLM) get fitted, but nothing reads them in this version, so all they cost is a little CPU time.

In [ ]:
%%writefile -a comp_stack.py
# ============================================================ 2. style n-grams + extra passages + gates
FUNCTION_WORDS = frozenset("a an the and or but if while although because as so than that which who whom whose this these those it its "
                           "they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with "
                           "without for of by over under between through into about is are was were be been being have has had do does did "
                           "can could will would shall should may might must not no nor all any some each every both either neither other "
                           "another such only also just very too more most less least here there when where why how what whether then now "
                           "still yet even much many few little".split())
TOKEN_RE = re.compile(r"\w+|[^\w\s]")


def shape_view(text):
    return re.sub(r"[^\W\d_]+", lambda m: 'X' if m[0].isupper() else 'x', text)


def function_view(text):
    return ' '.join(t.lower() if (t.lower() in FUNCTION_WORDS or not t.isalnum()) else 'CONTENT' for t in TOKEN_RE.findall(text))


class StyleGrams:
    """Four stacked TF-IDF views: char 3-5, word 1-3, shape (char 2-6 of the x/X text), function-word (word 1-3 of the masked text)."""
    VIEWS = [('char', str, (3, 5), 250_000, 1.0), ('word', str, (1, 3), 180_000, 0.7),
             ('char', shape_view, (2, 6), 60_000, 0.5), ('word', function_view, (1, 3), 50_000, 0.5)]

    def fit(self, texts):
        self.vecs = []
        for analyzer, view, ngr, maxf, w in self.VIEWS:
            opts = dict(analyzer=analyzer, ngram_range=ngr, lowercase=False, sublinear_tf=True, min_df=2, max_features=maxf, dtype=np.float64)
            if analyzer == 'word':
                opts['token_pattern'] = r"(?u)\b\w+\b|[^\w\s]"
            self.vecs.append((TfidfVectorizer(**opts).fit([view(t) for t in texts]), view, w))
        return self

    def transform(self, texts):
        return hstack([v.transform([view(t) for t in texts]) * w for v, view, w in self.vecs], format='csr')


def sparse_group_centre(X, groups, lo=GROUP_MIN, hi=GROUP_MAX):
    """Subtract each group's mean row (sparse); groups outside [lo, hi] are left as they are."""
    X = csr_matrix(X)
    _, inv, cnt = np.unique(np.asarray(groups), return_inverse=True, return_counts=True)
    keep = np.flatnonzero((cnt >= lo) & (cnt <= hi))
    if not len(keep):
        return X.copy()
    col = np.full(len(cnt), -1); col[keep] = np.arange(len(keep))
    rows = np.flatnonzero(col[inv] >= 0)
    M = csr_matrix((np.ones(len(rows)), (rows, col[inv][rows])), shape=(X.shape[0], len(keep)))
    means = diags(1.0 / cnt[keep]) @ (M.T @ X)
    return (X - M @ means).tocsr()


def svc_probs(svc, X):
    return full_prob(softmax_rows(svc.decision_function(X)), svc.classes_)


# ---- public in-style passages (prompt-bank protocol), one per (prompt, label); prompts answered by >= EXTRA_MIN_LABELS labels
def load_extra_passages():
    sources = [('**/ntl_extra_passages_claude.csv', 'model', None), ('**/gen_passages.csv', 'model', None),
               ('**/ntl_generations.csv', 'label', 'finish_reason'), ('**/ntl_proxy_generations.csv', 'label', None),
               ('**/ntl_generated*.csv', 'model', 'finish'), ('**/passages_clean.csv', 'model', None)]
    parts = []
    for pattern, label_col, finish_col in sources:
        for f in sorted(glob.glob('/kaggle/input/' + pattern, recursive=True)):
            d = pd.read_csv(f)
            if finish_col and finish_col in d:
                d = d[d[finish_col].eq('stop')]
            d = d.rename(columns={label_col: 'model'})[['prompt_id', 'model', 'text']].copy()
            d['prompt_id'] = ('pv_' if f.endswith('passages_clean.csv') else '') + d['prompt_id'].astype(str)   # prvsiyan's prompts are a different bank
            parts.append(d)
    if not parts:
        return pd.DataFrame(columns=['prompt_id', 'model', 'text'])
    x = pd.concat(parts, ignore_index=True)
    x = x[x['model'].isin(LABELS)].copy()
    x['text'] = x['text'].astype(str).map(lambda t: re.sub(r'\s+', ' ', re.sub(r'^\*\*(.*)\*\*$', r'\1', t.strip())).strip())
    x = x[x['text'].str.split().str.len().between(30, 300)].drop_duplicates(subset=['model', 'text'])
    x = x.drop_duplicates(subset=['prompt_id', 'model'])
    x = x[x.groupby('prompt_id')['model'].transform('nunique') >= EXTRA_MIN_LABELS].reset_index(drop=True)
    return x


EXTRA = load_extra_passages()
EX_T, EX_Y, EX_G = EXTRA['text'].tolist(), EXTRA['model'].map(LAB2I).to_numpy(dtype=int), EXTRA['prompt_id'].to_numpy(dtype=object)
print(f'extra passages: {len(EXTRA)} rows, {EXTRA["prompt_id"].nunique()} prompts', flush=True)


def extra_sample_weights():
    """Per class: original rows keep (1 - EXTRA_SHARE) of the class weight, the class's extra rows share EXTRA_SHARE of it."""
    n_tr, n_ex = np.bincount(y, minlength=K).astype(float), np.bincount(EX_Y, minlength=K).astype(float)
    w_tr = np.where(n_ex[y] > 0, 1.0 - EXTRA_SHARE, 1.0)
    w_ex = EXTRA_SHARE * n_tr[EX_Y] / np.maximum(n_ex[EX_Y], 1.0)
    return np.concatenate([w_tr, w_ex])


grams = StyleGrams().fit(TR + EX_T)                       # vocabulary and idf from train + extra
X_tr, X_ex, X_te = grams.transform(TR), grams.transform(EX_T) if len(EX_T) else None, grams.transform(TE)


def fit_svc(X, labels, C=0.5, **kw):
    return LinearSVC(C=C, random_state=42, max_iter=4000).fit(X, labels, **kw)


if X_ex is not None:
    svc_plain = fit_svc(vstack([X_tr, X_ex]).tocsr(), np.concatenate([y, EX_Y]), sample_weight=extra_sample_weights())
else:
    svc_plain = fit_svc(X_tr, y)
P_style = svc_probs(svc_plain, X_te)

# centred twin: prompt-centred training rows (extra rows centred within their own prompts) -> inferred test groups
X_tr_c = sparse_group_centre(X_tr, PROMPT)
if X_ex is not None:
    svc_centred = fit_svc(vstack([X_tr_c, sparse_group_centre(X_ex, EX_G, GROUP_MIN, 10 ** 6)]).tocsr(), np.concatenate([y, EX_Y]),
                          sample_weight=extra_sample_weights())
else:
    svc_centred = fit_svc(X_tr_c, y)
G_style = normalised_content_groups(TE)
rows_style = eligible(G_style)
describe_groups('stylegram groups', G_style)
if rows_style.any():
    P_style[rows_style] = svc_probs(svc_centred, sparse_group_centre(X_te[rows_style], G_style[rows_style]))
if TINY:   # exercise the centred path on training rows grouped by their true prompt (output discarded)
    st_idx, st_g = self_test_rows()
    acc = np.mean(svc_centred.classes_[svc_centred.decision_function(sparse_group_centre(X_tr[st_idx], st_g)).argmax(1)] == y[st_idx])
    print(f'  centred stylegram self-test on {len(st_idx)} training rows (seen in training): acc {acc:.2f}', flush=True)
add_member('stylegram', P_style)
del X_tr_c, svc_plain, svc_centred, P_style


# ---- gates: style SVCs that only re-rank inside one family after the blend
def family_gate(texts, labels, classes, C, sample_weight=None, balanced=False):
    g = StyleGrams().fit(texts)
    svc = LinearSVC(C=C, class_weight='balanced' if balanced else None, random_state=42, max_iter=4000).fit(
        g.transform(texts), labels, sample_weight=sample_weight)
    assert list(svc.classes_) == list(classes)
    d = svc.decision_function(g.transform(TE))
    return np.column_stack([-d, d]) if len(classes) == 2 else d


GATES = {}
api_claude = sorted(glob.glob('/kaggle/input/**/ntl_extra_passages_claude.csv', recursive=True))
if api_claude:
    api = pd.read_csv(api_claude[0])
    m = np.isin(y, FAMILY_IDX['claude'])
    GATES['claude'] = family_gate([t for t, k in zip(TR, m) if k] + api['text'].astype(str).tolist(),
                                  np.concatenate([y[m], api['model'].map(LAB2I).to_numpy(dtype=int)]), FAMILY_IDX['claude'], 0.5)
    print(f'claude gate: {int(m.sum())} official + {len(api)} API rows', flush=True)
    del api
else:
    print('Claude API passages not attached: Claude gate off', flush=True)
for fam, C in [('gpt', 1.0), ('gemini', 1.0), ('glm', 0.5)]:
    m, me = np.isin(y, FAMILY_IDX[fam]), np.isin(EX_Y, FAMILY_IDX[fam])
    GATES[fam] = family_gate([t for t, k in zip(TR, m) if k] + [t for t, k in zip(EX_T, me) if k], np.concatenate([y[m], EX_Y[me]]),
                             FAMILY_IDX[fam], C, sample_weight=np.concatenate([np.ones(int(m.sum())), np.full(int(me.sum()), 0.25)]), balanced=True)
    print(f'{fam} gate: {int(m.sum())} official + {int(me.sum())} extra rows', flush=True)
del grams, X_tr, X_ex, X_te



### 3. Stylometry + LightGBM

Hand-made counts for each passage: sentence and word lengths, vocabulary richness, capitals, digits, non-ASCII characters, the rate of every punctuation mark, a few word endings (-ly, -ing, -tion) and the rate of every English stop word. A small LightGBM sits on top. Its weight in the blend is small (0.063), but it looks at the text in a completely different way from everything else here.

In [ ]:
%%writefile -a comp_stack.py
# ============================================================ 3. stylometric features + LightGBM
PUNCT = list(',.;:!?()[]"\'-') + ['—', '–', '‘', '’', '“', '”', '…', '*', '/', '%']
STOPS = sorted(ENGLISH_STOP_WORDS)
SENT_RE, WORD_RE = re.compile(r'(?<=[.!?])\s+'), re.compile(r"[A-Za-z’']+")


def stylometry(t):
    words = WORD_RE.findall(t)
    low = [w.lower() for w in words]
    nw, nc = max(len(words), 1), max(len(t), 1)
    sents = [s for s in SENT_RE.split(t.strip()) if s]
    sl = np.array([len(WORD_RE.findall(s)) for s in sents] or [0], dtype=float)
    wl = np.array([len(w) for w in words] or [0], dtype=float)
    f = [np.log1p(nw), len(sents), sl.mean(), sl.std(), sl.max(), sl.min(), wl.mean(), wl.std(), (wl >= 9).mean(), len(set(low)) / nw,
         sum(c.isupper() for c in t) / nc, sum(c.isdigit() for c in t) / nc, sum(ord(c) > 127 for c in t) / nc,
         t.count(', ') / nw, t.count(' and ') / nw, t.count(' but ') / nw,
         sum(1 for s in sents if s[:1].isupper() and s.split()[0].lower() in ('this', 'these', 'it', 'they')) / max(len(sents), 1),
         sum(w.endswith('ly') for w in words) / nw, sum(w.endswith('ing') for w in words) / nw, sum(w.endswith('tion') for w in words) / nw]
    f += [t.count(q) / nw for q in PUNCT]
    counts = pd.Series(low).value_counts() if low else pd.Series(dtype=float)
    f += [counts.get(s, 0) / nw for s in STOPS]
    return f


F_tr, F_te = (np.array([stylometry(t) for t in txt], dtype=np.float32) for txt in (TR, TE))
try:
    import lightgbm as lgb
    booster = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=10, subsample=0.8, subsample_freq=1,
                                 colsample_bytree=0.5, reg_lambda=1.0, random_state=0, verbose=-1, n_jobs=4).fit(F_tr, y)
    add_member('stylometry_lgb', full_prob(booster.predict_proba(F_te), booster.classes_))
    del booster
except ImportError:
    assert SMOKE, 'lightgbm missing'
    print('smoke: lightgbm unavailable, stylometry member skipped', flush=True)
del F_tr, F_te



### 4b. The shared-prompt Gaussian, again

The same code as `gauss.py` above (I checked: identical), pasted into the stack so the script is self-contained. Nothing new in this section.

In [ ]:
%%writefile -a comp_stack.py
# ============================================================ 4b. shared-prompt Gaussian group head
# Verbatim core of our research tool tools/group_gaussian.py (sha256 8a42427b77a37f56...), numpy only.
# Model per dense view, in a reduced space z (standardise -> within-prompt-residual PCA, fitted on training rows):
#   z_i = mu_{y_i} + p_g + e_i,  p_g ~ N(0, S_p) shared by every passage of prompt group g,  e_i ~ N(0, S_e).
# A joint diagonalisation makes the dimensions independent; for a detected group of n passages the exact joint
# log-likelihood of a label tuple is sum_i a_i(c_i) + sum_{i<j} b_ij(c_i, c_j). Groups of 2-3 are decoded exactly over
# all label tuples with the at-most-once constraint; larger groups by a cavity EM with the finite-population-corrected
# prompt-effect estimate lam / (1 + (n-1) lam); singletons integrate the prompt effect out (never used here).
import itertools
K = 18
DEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,
           "iters": 5, "repel": True, "sp_scale": 1.0}

# =================================================================== primitives ====
def _lse(A, axis=None, keepdims=False):
    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
    if keepdims: return out
    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

def log_softmax(L): return L - _lse(L, axis=1, keepdims=True)

def _lw_shrinkage(X):
    """Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered)."""
    from sklearn.covariance import ledoit_wolf_shrinkage
    return float(ledoit_wolf_shrinkage(X, assume_centered=True))

def _shrunk(S, a):
    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)

def _two_way(Z, y, pinv, G, iters=4):
    """additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds)."""
    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)
    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))
    for _ in range(iters):
        R = Z - P[pinv]
        for c in cls: mu[c] = R[y == c].mean(0)
        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]
    return mu, P, cnt_p

def fit(X, y, prompts, cfg=None):
    """fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.
    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller's cross-fit)."""
    c = dict(DEFAULT); c.update(cfg or {})
    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape
    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)
    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx
    if c["red"] == "none":
        V = np.eye(D)
    else:
        if c["red"] == "wpca":
            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]
        elif c["red"] == "pca": R = Xs
        else: raise ValueError(c["red"])
        d = int(min(c["d"], N - G - 1, D))
        # economy SVD through the (N x N) Gram matrix (N << D for the real views)
        if N < D:
            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]
            V = (R.T @ U) / np.sqrt(w)
        else:
            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T
    Z = Xs @ V; d = Z.shape[1]
    mu, P, ng = _two_way(Z, y, pinv, G)
    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)
    Se_raw = E.T @ E / dof
    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)
    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])
    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]
    # joint diagonalisation: A Se A' = I, A Sp A' = diag(lam)
    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)
    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li
    W = V @ A.T
    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,
            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}

def transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]

_MASK = {}
def _inj_mask(n, C):
    if (n, C) not in _MASK:
        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)
        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]
        _MASK[(n, C)] = bad
    return _MASK[(n, C)]

def singleton_ll(U, m, lam):
    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""
    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)

def joint_terms(U, m, lam):
    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""
    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]
    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)
    return a, B

def joint_table(a, B):
    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""
    n, C = a.shape; Q = np.zeros([C] * n)
    for i in range(n):
        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)
    for i, j in itertools.combinations(range(n), 2):
        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)
    return Q

def exact_marginals(Q, T=1.0, inj=True):
    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""
    n = Q.ndim; C = Q.shape[0]; Qt = Q / T
    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)
    lz = _lse(Qt); out = np.empty((n, C))
    for i in range(n):
        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz
    return out

def em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):
    """cavity EM for one group (any n >= 2): returns (n, C) log q."""
    n = len(U); v = lam / (1 + (n - 1) * lam)
    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))
    q = np.exp(log_softmax(L))
    for _ in range(iters):
        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v                  # cavity prompt-effect means (n, d)
        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T
        if repel:   # mean-field at-most-once: discourage classes the other members already hold
            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)
        q = np.exp(log_softmax(L))
    return log_softmax(L)

LAM_DIFF = 1e6
def lam_g(prm, cfg=None):
    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg 'diff'): then the joint likelihood
    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs
    centred prototypes) and the prompt term cancels exactly."""
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})
    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]

def group_logpost(U, prm, T1=None, Tg=None, cfg=None):
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)
    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg
    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)
    if n <= c["n_exact"]:
        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])
    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])

def scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):
    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""
    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))
    sizes = np.bincount(lab); s1 = sizes[lab] == 1
    T1 = prm["T1"] if T1 is None else T1
    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)
    for g in np.where(sizes >= 2)[0]:
        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)
    return out



### 4. Frozen models: plain heads, centred twins, Gaussian heads

For each frozen model: load only the layers it needs, pool the hidden states of two chosen layers over the passage, and fit simple heads on the training rows.

- `plain_and_twin`: a logistic regression on standardised features (the test side is standardised with its own statistics), plus a centred twin fit on prompt-centred features and applied to the grouped test rows.
- `gaussian_group_head`: the shared-prompt Gaussian on one layer each of four of these models.

GLM-4-9B runs first, because its layer-24 features build the test groups that everything after it uses.

One change of mine is in `find_checkpoint`: it now skips folders that have a `config.json` but no weights. lightningv08's assets contain such a folder (a LoRA checkpoint's `base_config/`), and without the check the search for a Qwen3 base model could land on it.

In [ ]:
%%writefile -a comp_stack.py
# ============================================================ 4. frozen models: plain heads, centred twins, Gaussian group heads
if not SMOKE:
    from transformers import AutoConfig, AutoModel, AutoTokenizer
CACHED_PARTS = {}   # key -> {part name: (train features, test features)}


def find_checkpoint(model_type, hidden=None, dir_name=None):
    """A checkpoint directory under /kaggle/input by model_type (+ hidden size) or by directory name."""
    pattern = f'/kaggle/input/**/{dir_name}/config.json' if dir_name else '/kaggle/input/**/config.json'
    for cfg in sorted(glob.glob(pattern, recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if dir_name or (c.get('model_type') == model_type and (hidden is None or c.get('hidden_size') == hidden)
                        and glob.glob(os.path.join(os.path.dirname(cfg), '*.safetensors'))):   # fork: skip config-only dirs (a LoRA checkpoint's base_config/)
            return os.path.dirname(cfg)
    return None


def load_truncated_decoder(path, keep, dtype):
    cfg = AutoConfig.from_pretrained(path)
    cfg.num_hidden_layers = keep
    if getattr(cfg, 'layer_types', None):
        cfg.layer_types = cfg.layer_types[:keep]
    kw = dict(config=cfg, dtype=dtype)
    if DEV == 'cuda':
        kw['device_map'] = 'auto'
    return AutoModel.from_pretrained(path, **kw).eval()


def style_token_mask(tok):
    """Vocabulary mask of 'style' tokens: the decoded piece is an English stop word or pure punctuation."""
    pieces = [tok.decode([i]) for i in range(len(tok))]
    return torch.tensor([(p.strip().lower() in ENGLISH_STOP_WORDS) or bool(re.fullmatch(r'\s*[^\w\s]+\s*', p)) for p in pieces])


@torch.no_grad()
def pooled_states(model, tok, texts, parts, decoder, bs, max_len=512):
    """{'L{layer}_{pool}': (n, hidden)} masked mean pools of hidden_states[layer] (all tokens, or style tokens only)."""
    H = model.config.hidden_size
    out = {f'L{L}_{pool}': np.zeros((len(texts), H), dtype=np.float32) for L, pool in parts}
    smask = style_token_mask(tok) if any(pool == 'style' for _, pool in parts) else None
    dev = next(model.parameters()).device
    order = np.argsort([len(t) for t in texts])
    retried = 0
    for b0 in range(0, len(texts), bs):
        idx = order[b0:b0 + bs]
        if decoder:
            batch = [(tok.bos_token or '') + texts[i] for i in idx]
            enc = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=max_len, add_special_tokens=False).to(dev)
        else:
            enc = tok([texts[i] for i in idx], return_tensors='pt', padding=True, truncation=True, max_length=max_len).to(dev)
        hs = model(**enc, output_hidden_states=True).hidden_states
        if decoder and not all(torch.isfinite(hs[L]).all() for L, _ in parts):   # fp16 overflow: redo the batch in bf16
            retried += 1
            model.to(torch.bfloat16)
            hs = model(**enc, output_hidden_states=True).hidden_states
        m_all = enc['attention_mask'].float()
        for L, pool in parts:
            m = m_all * smask.to(m_all.device)[enc['input_ids']].float() if pool == 'style' else m_all
            m = m.unsqueeze(-1).to(hs[L].device)
            out[f'L{L}_{pool}'][idx] = ((hs[L].float() * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
        del hs, enc
    if retried:
        print(f'  {retried} batches redone in bf16', flush=True)
    return {k: np.nan_to_num(v) for k, v in out.items()}


def pseudo_parts(parts, dim=48):
    """SMOKE only: deterministic stand-ins for the model passes (TF-IDF SVD of the texts), same downstream code."""
    from sklearn.decomposition import TruncatedSVD
    tf = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), sublinear_tf=True, min_df=2).fit(TR + TE)
    out = {}
    for j, (L, pool) in enumerate(parts):
        svd = TruncatedSVD(dim, random_state=L + 7 * j).fit(tf.transform(TR))
        out[f'L{L}_{pool}'] = (svd.transform(tf.transform(TR)).astype(np.float32), svd.transform(tf.transform(TE)).astype(np.float32))
    return out


def extract_parts(key, parts, loader):
    """Train and test pools for every requested part; loader() returns (model, tok, decoder flag, batch size)."""
    if SMOKE:
        return pseudo_parts(parts)
    model, tok, decoder, bs = loader()
    tr, te = pooled_states(model, tok, TR, parts, decoder, bs), pooled_states(model, tok, TE, parts, decoder, bs)
    del model, tok
    gc.collect()
    if DEV == 'cuda':
        torch.cuda.empty_cache()
    return {k: (tr[k], te[k]) for k in tr}


def decoder_loader(model_type, hidden, keep):
    def load():
        path = find_checkpoint(model_type, hidden)
        assert path, f'no {model_type} checkpoint with hidden size {hidden} under /kaggle/input'
        tok = AutoTokenizer.from_pretrained(path)
        tok.padding_side = 'right'
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        return load_truncated_decoder(path, keep, torch.float16 if DEV == 'cuda' else torch.float32), tok, True, 8
    return load


def encoder_loader(dir_name):
    def load():
        path = find_checkpoint(None, dir_name=dir_name)
        assert path, f'encoder directory {dir_name} not found under /kaggle/input'
        return AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval(), AutoTokenizer.from_pretrained(path), False, 16
    return load


def concat_parts(cache, parts):
    return (np.concatenate([cache[f'L{L}_{p}'][0] for L, p in parts], 1), np.concatenate([cache[f'L{L}_{p}'][1] for L, p in parts], 1))


def plain_and_twin(key, Etr, Ete, C):
    """Plain head (test-side standardisation) for the blend; centred twin evidence on grouped rows."""
    if W[key] > 0:
        sc = StandardScaler().fit(Etr)
        clf = LogisticRegression(C=C, max_iter=5000).fit(sc.transform(Etr), y)
        Xte = (Ete - Ete.mean(0)) / (Ete.std(0) + 1e-6) if len(Ete) >= 2 else sc.transform(Ete)
        add_member(key, full_prob(clf.predict_proba(Xte), clf.classes_))
    else:
        print(f'  {key}: plain head has weight 0 in the 0.563 blend, not fitted', flush=True)
    if key in TWIN_W:
        csc = StandardScaler().fit(group_centre_dense(Etr, PROMPT))
        twin = LogisticRegression(C=C, max_iter=5000).fit(csc.transform(group_centre_dense(Etr, PROMPT)), y)
        if GROUP_ROWS.any():
            pc = twin.predict_proba(csc.transform(group_centre_dense(Ete[GROUP_ROWS], G_TE[GROUP_ROWS])))
            add_member(key + '_twin', full_prob(pc, twin.classes_), rows=GROUP_ROWS, weight=TWIN_W[key])
        if TINY:
            st_idx, st_g = self_test_rows()
            ps = twin.predict_proba(csc.transform(group_centre_dense(Etr[st_idx], st_g)))
            print(f'  {key} twin self-test on {len(st_idx)} training rows (seen in training): acc {np.mean(twin.classes_[ps.argmax(1)] == y[st_idx]):.2f}', flush=True)


def gaussian_group_head(name, spec, Xtr, Xte):
    """Our addition: fit the shared-prompt Gaussian on the training prompts, score the grouped test rows jointly within their group."""
    prm = fit(Xtr, y, PROMPT, dict(GAUSS_SETTING, d=spec['d']))
    prm['T1'], prm['Tg'] = spec['T1'], spec['Tg']
    print(f'  {name}: d {prm["info"]["d"]}, shrinkage e/p {prm["info"]["a_e"]}/{prm["info"]["a_p"]}, lam pct {prm["info"]["lam_pct"]}', flush=True)
    if GROUP_ROWS.any():
        LP = scores(Xte[GROUP_ROWS], np.unique(G_TE[GROUP_ROWS], return_inverse=True)[1], prm)
        add_member(name, np.exp(LP), rows=GROUP_ROWS, weight=spec['w'])
    if TINY:
        st_idx, st_g = self_test_rows()
        LP = scores(Xtr[st_idx], st_g, prm)
        print(f'  {name} self-test on {len(st_idx)} training rows (seen in training): acc {np.mean(LP.argmax(1) == y[st_idx]):.2f}', flush=True)


for key, model_type, hidden, parts, extra_parts, C in LM_HEADS:
    print(f'[{key}] {model_type} hidden {hidden} parts {parts + extra_parts}', flush=True)
    cache = extract_parts(key, parts + extra_parts, decoder_loader(model_type, hidden, max(L for L, _ in parts + extra_parts) + 1))
    if G_TE is None:   # the GLM pass comes first: its layer-24 pool defines the test groups
        G_TE = mixed_groups(TE, cache[f'L{GROUP_LAYER}_mean'][1])
        GROUP_ROWS = eligible(G_TE)
        describe_groups('test groups (content + GLM L24)', G_TE)
        if SMOKE:
            same_g, same_p = G_TE[:, None] == G_TE[None, :], test['prompt_id'].to_numpy(dtype=object)[:, None] == test['prompt_id'].to_numpy(dtype=object)[None, :]
            off = ~np.eye(len(TE), dtype=bool)
            print(f'  smoke grouping: pair precision {(same_g & same_p & off).sum() / max((same_g & off).sum(), 1):.2f} '
                  f'recall {(same_g & same_p & off).sum() / max((same_p & off).sum(), 1):.2f}', flush=True)
    CACHED_PARTS[key] = cache
    plain_and_twin(key, *concat_parts(cache, parts), C)
    for name, spec in GAUSS_HEADS.items():
        if spec['src'] == key:
            gaussian_group_head(name, spec, *cache[spec['part']])
    del cache
    CACHED_PARTS.pop(key)
    gc.collect()

for key, dir_name, parts, C in ENC_HEADS:
    print(f'[{key}] {dir_name} parts {parts}', flush=True)
    cache = extract_parts(key, parts, encoder_loader(dir_name))
    plain_and_twin(key, *concat_parts(cache, parts), C)
    for name, spec in GAUSS_HEADS.items():
        if spec['src'] == key:
            gaussian_group_head(name, spec, *cache[spec['part']])
    del cache
    gc.collect()



### 5. LoRA classifiers (inference only)

Four LoRA checkpoints trained by other people: three on Qwen3-8B (24 layers kept) and one on Qwen2.5-7B (20 layers). One trunk is loaded per base model and the adapters are swapped in turn. Each checkpoint ships its readouts (plain, centred, and for one of them a within-Claude head) and the features of a few training rows. The script recomputes those features and only trusts the checkpoint if they match within 2%.

`lora24_public` is the member I brought back. If its features don't match, it skips itself instead of failing the whole stack. It also saves its raw test features to `/tmp/night24_raw.npy` for the spectral heads.

In [ ]:
%%writefile -a comp_stack.py
# ============================================================ 5. LoRA classifiers (inference only)
@torch.no_grad()
def score_lora_checkpoints(trunk_spec, pool_layers, checkpoints):
    """One truncated trunk, adapters swapped in turn. Plain readout -> blend member; centred readout -> twin on grouped rows;
    within-Claude readout -> Claude gate."""
    import importlib.util as iu
    _find = iu.find_spec   # Kaggle's torchao build is rejected by peft; hide it (unused here)
    iu.find_spec = lambda name, *a, **k: None if name.split('.')[0] == 'torchao' else _find(name, *a, **k)
    from peft import PeftModel
    import peft.import_utils as pi
    pi.is_torchao_available = lambda *a, **k: False
    from safetensors.torch import load_file
    model_type, H, keep = trunk_spec
    base = find_checkpoint(model_type, H)
    assert base, f'no {model_type} base checkpoint under /kaggle/input'
    cfg = AutoConfig.from_pretrained(base)
    cfg.num_hidden_layers, cfg.use_cache = keep, False
    if getattr(cfg, 'layer_types', None) is not None:
        cfg.layer_types = cfg.layer_types[:keep]
    trunk = AutoModel.from_pretrained(base, config=cfg, dtype=torch.float16, device_map='auto').eval()
    for key, marker, norms_file, anchor_file in checkpoints:
        hits = sorted(glob.glob('/kaggle/input/' + marker, recursive=True))
        if not hits:   # upstream training kernel no longer public: drop this member, keep the rest of the stack
            print(f'  SKIP {key}: checkpoint not attached ({marker})', flush=True); continue
        ck = pathlib.Path(os.path.dirname(hits[0]))
        print(f'[{key}] {ck}', flush=True)
        tok = AutoTokenizer.from_pretrained(ck / 'tokenizer')
        tok.padding_side = 'right'
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        lm = PeftModel.from_pretrained(trunk, ck / 'adapter').eval()
        state = load_file(str(ck / norms_file))
        norms = torch.nn.ModuleList([torch.nn.LayerNorm(H) for _ in pool_layers])
        norms.load_state_dict({k[len('norms.'):]: v for k, v in state.items() if k.startswith('norms.')})
        ro = np.load(ck / 'readouts.npz')
        assert list(ro['classes']) == LABELS and list(ro['pool_layers']) == pool_layers
        dev = lm.get_input_embeddings().weight.device
        norms = norms.to(dev).float()

        def token_ids(texts):
            return [ids + [tok.eos_token_id] for ids in tok(list(texts), add_special_tokens=False, truncation=True, max_length=LORA_MAX_LEN)['input_ids']]

        def features(texts, bs):
            ids = token_ids(texts)
            Z = np.zeros((len(ids), H * len(pool_layers)), dtype=np.float32)
            order = np.argsort([len(t) for t in ids], kind='stable')
            for b0 in range(0, len(order), bs):
                idx = order[b0:b0 + bs]
                batch = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to(dev)
                with torch.autocast('cuda', dtype=torch.float16):
                    hs = lm(**batch, output_hidden_states=True, use_cache=False).hidden_states
                m = batch['attention_mask'].unsqueeze(-1).float()
                m[:, 0] = 0   # the first token is masked out of the mean, as in training
                Z[idx] = torch.cat([nm((hs[L].float() * m.to(hs[L].device)).sum(1).to(dev) / m.sum(1).to(dev))
                                    for L, nm in zip(pool_layers, norms)], -1).cpu().numpy()
                del hs, batch
            return Z

        if (ck / anchor_file).exists():   # the checkpoint's own features for training rows 0..2: our forward pass must reproduce them
            saved = np.load(ck / anchor_file)['features'].astype(np.float32)
            ours = features(TR[:3], 3)
            rel = float(np.abs(ours - saved).mean() / (np.abs(saved).mean() + 1e-9))
            print(f'  parity with the checkpoint anchors: max abs {float(np.abs(ours - saved).max()):.4f}, mean rel {rel:.5f}', flush=True)
            if rel >= 0.02 and key == 'lora24_public':   # fork: the restored member must never take the stack down
                print(f'  SKIP {key}: features do not match the checkpoint', flush=True)
                trunk = lm.unload(); del lm; gc.collect(); continue
            assert rel < 0.02, f'{key}: our features do not match the checkpoint'
        Z = features(TE, LORA_BS)
        if key == 'lora24_public' and os.environ.get('NTL_NIGHT24_RAW'):   # fork: raw features for Kirill's spectral readout heads
            np.save(os.environ['NTL_NIGHT24_RAW'], Z)
        add_member(key, softmax_rows(Z @ ro['plain_coef'].T + ro['plain_intercept']))
        if 'claude_coef' in ro.files:
            lc = Z @ ro['claude_coef'].T + ro['claude_intercept']
            GATES['claude_lora'] = lc - np.log(np.exp(lc - lc.max(1, keepdims=True)).sum(1, keepdims=True)) - lc.max(1, keepdims=True)
        if key in LORA_TWIN_W and GROUP_ROWS.any():
            Zc = group_centre_dense(Z[GROUP_ROWS], G_TE[GROUP_ROWS])
            add_member(key + '_twin', softmax_rows(Zc @ ro['center_coef'].T + ro['center_intercept']), rows=GROUP_ROWS, weight=LORA_TWIN_W[key])
        trunk = lm.unload()
        del lm, Z
        gc.collect()
        torch.cuda.empty_cache()
    del trunk
    gc.collect()
    torch.cuda.empty_cache()


if not SMOKE:
    for trunk_spec, pool_layers, checkpoints in LORA_RUNS:
        score_lora_checkpoints(trunk_spec, pool_layers, checkpoints)



### 6. Token-attention head, UltraBERT, and the output

- The token-attention head (from Lucas Boesen's training notebooks) puts a small attention pooling on top of DeBERTa's token states.
- UltraBERT-380M (pjmathematician) is a fine-tuned encoder; its five folds are averaged.

The script ends by saving the blend as `comp_logit.npy`, with the label order and test ids next to it in `comp_meta.json`, so the main pipeline can check that it reads the rows in the right order.

In [ ]:
%%writefile -a comp_stack.py
# ============================================================ 6. token-attention heads + UltraBERT
class TokenAttention(torch.nn.Module if torch is not None else object):   # the module of the public training kernels (lucasboesen/name-that-llm-train-tokattn, -tokattn2)
    def __init__(self, k, h=4096):
        super().__init__()
        self.drop, self.proj, self.score, self.out = torch.nn.Dropout(0.2), torch.nn.Linear(h, 64), torch.nn.Linear(64, 1), torch.nn.Linear(128, k)

    def forward(self, x, m):
        h = torch.tanh(self.proj(self.drop(x)))
        a = torch.softmax(self.score(h).squeeze(-1).masked_fill(~m, -1e4), -1).unsqueeze(-1)
        mf = m.unsqueeze(-1).float()
        return self.out(self.drop(torch.cat([(h * mf).sum(1) / mf.sum(1).clamp(min=1), (a * h).sum(1)], -1)))


@torch.no_grad()
def token_attention_probs(texts, head_file, backbone, bs=8):
    hits = sorted(glob.glob('/kaggle/input/**/' + head_file, recursive=True))
    assert hits, f'training-kernel output holding {head_file} not attached'
    d = torch.load(hits[0], map_location='cpu')
    assert list(d['classes']) == LABELS
    decoder = d.get('decoder', True)
    path = find_checkpoint('qwen3_5') if backbone == 'qwen3_5' else find_checkpoint(None, dir_name='deberta-v3-large')
    assert path, f'backbone {backbone} not attached'
    cfg = AutoConfig.from_pretrained(path)
    if decoder:
        tc = getattr(cfg, 'text_config', None) or cfg
        tc.num_hidden_layers = d['keep']
        if getattr(tc, 'layer_types', None):
            tc.layer_types = tc.layer_types[:d['keep']]
        lm = AutoModel.from_pretrained(path, config=cfg, dtype=torch.float16, device_map='auto').eval()
    else:
        lm = AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval()
    tok = AutoTokenizer.from_pretrained(path)
    tok.padding_side = 'right'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    H = d.get('hidden', 4096)
    net = TokenAttention(K, H)
    net.load_state_dict(d['state_dict'])
    dev = lm.get_input_embeddings().weight.device
    net, mu, sd = net.to(dev).eval(), d['mu'].float().to(dev), d['sd'].float().to(dev)
    ids = tok(list(texts), add_special_tokens=not decoder, truncation=True, max_length=d['max_len'])['input_ids']
    P = np.zeros((len(texts), K))
    order = np.argsort([len(t) for t in ids])
    for b0 in range(0, len(order), bs):
        idx = order[b0:b0 + bs]
        batch = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to(dev)
        out = lm(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], output_hidden_states=True)
        hs = (out.hidden_states if getattr(out, 'hidden_states', None) is not None else out.language_model_outputs.hidden_states)[d['layer']]
        n = batch['attention_mask'].sum(1)
        T = max(int(n.max()) - 1, 1)
        x = torch.zeros(len(idx), T, H, device=dev)
        m = torch.zeros(len(idx), T, dtype=torch.bool, device=dev)
        for r in range(len(idx)):   # first token dropped, states rounded through fp16, as in training
            k = int(n[r]) - 1
            if k > 0:
                x[r, :k] = ((torch.nan_to_num(hs[r, 1:k + 1].to(dev).float()).half().float() - mu) / sd).clamp(-8, 8)
                m[r, :k] = True
        P[idx] = torch.softmax(net(x, m), -1).cpu().numpy()
        del hs, out, batch
    del lm
    gc.collect()
    torch.cuda.empty_cache()
    return P


@torch.no_grad()
def ultrabert_probs(texts):
    from transformers import AutoModelForSequenceClassification
    folds = None
    for root in glob.glob('/kaggle/input/**/ultrabert-380m-ntl', recursive=True):
        fs = [f for f in sorted(glob.glob(os.path.join(root, 'fold*'))) if os.path.exists(os.path.join(f, 'config.json'))
              and json.load(open(os.path.join(f, 'config.json'))).get('model_type') == 'ultrabert']
        if len(fs) == 5:
            folds = fs
            break
    assert folds, 'ultrabert-380m-ntl (5 folds) not attached'
    tok = AutoTokenizer.from_pretrained(folds[0], trust_remote_code=True)
    ids = tok(texts, truncation=True, max_length=512)['input_ids']
    order = np.argsort([len(e) for e in ids])
    P = np.zeros((len(texts), K))
    for fp in folds:
        em = AutoModelForSequenceClassification.from_pretrained(fp, trust_remote_code=True, dtype=torch.float16, attn_implementation='sdpa').to(DEV).eval()
        cols = [LAB2I[em.config.id2label[i]] for i in range(len(em.config.id2label))]
        for b0 in range(0, len(order), 32):
            idx = order[b0:b0 + 32]
            b = tok.pad({'input_ids': [ids[j] for j in idx]}, return_tensors='pt').to(DEV)
            P[np.ix_(idx, cols)] += em(**b).logits.float().softmax(-1).cpu().numpy() / len(folds)
        del em
        gc.collect()
        torch.cuda.empty_cache()
    return P


if not SMOKE:
    for key, head_file, backbone in TOKATTN_HEADS:
        if TINY:   # in-sample check on training rows the head was fitted on: must be high, or the checkpoint is not being reproduced
            acc = float(np.mean(token_attention_probs(TR[:40], head_file, backbone).argmax(1) == y[:40]))
            print(f'  {key} in-sample check on 40 training rows: acc {acc:.2f}', flush=True)
            assert acc > 0.5, f'{key} does not reproduce its training fit'
        add_member(key, token_attention_probs(TE, head_file, backbone))
    add_member('ultrabert', ultrabert_probs(TE))



def _np_logsoftmax(z):
    from scipy.special import logsumexp as _lse_np
    z = np.asarray(z, dtype=np.float64)
    return z - _lse_np(z, axis=1, keepdims=True)

def _apply_conditional_family_gate(gate_key, family, lam, scores_are_logprob=False):
    '''Use the already-trained family gate only to rerank siblings, preserving total family mass.
    Confidence-aware interpolation avoids overriding high-confidence cross-family decisions.'''
    score = GATES.get(gate_key)
    if score is None:
        print(f'  family gate skipped {gate_key}: no fitted scores', flush=True)
        return
    idx = FAMILY_IDX[family]
    score = np.asarray(score, dtype=np.float64)
    if score.shape != (len(TE), len(idx)):
        print(f'  family gate skipped {gate_key}: shape {score.shape} expected {(len(TE), len(idx))}', flush=True)
        return
    if scores_are_logprob:
        gate_lp = _np_logsoftmax(score)
    else:
        score = score - score.mean(axis=1, keepdims=True)
        scale = np.sqrt(np.mean(score * score, axis=1, keepdims=True))
        score = score / np.maximum(scale, 0.25)
        gate_lp = _np_logsoftmax(score)
    base = LOGIT[:, idx].copy()
    base_lp = _np_logsoftmax(base)
    base_p = np.exp(base_lp)
    sp = np.sort(base_p, axis=1)
    margin = sp[:, -1] - sp[:, -2]
    confidence_gate = np.clip((0.45 - margin) / 0.45, 0.0, 1.0)
    strength = min(0.45, float(lam) / (1.0 + float(lam))) * confidence_gate
    mixed_lp = _np_logsoftmax((1.0 - strength[:, None]) * base_lp + strength[:, None] * gate_lp)
    # Preserve the ensemble's probability mass assigned to this family; only adjust within-family identity.
    from scipy.special import logsumexp as _lse_np
    family_mass = _lse_np(base, axis=1)
    LOGIT[:, idx] = family_mass[:, None] + mixed_lp
    print(f'  family gate applied {gate_key}: mean strength={strength.mean():.4f}, active rows={int((strength > 0).sum())}', flush=True)

# These gates were fitted above but were previously never added to LOGIT. Apply them conservatively.
if not SMOKE:
    _apply_conditional_family_gate('claude', 'claude', GATE_LAM['claude_style'])
    _apply_conditional_family_gate('claude_lora', 'claude', GATE_LAM['claude_lora'], scores_are_logprob=True)
    for _fam in ('gpt', 'gemini', 'glm'):
        _apply_conditional_family_gate(_fam, _fam, GATE_LAM['family'])


np.save('/kaggle/working/comp_logit.npy', LOGIT)
json.dump({'labels': LABELS, 'ids': test['id'].astype(str).tolist(), 'seconds': time.time() - T0}, open('/kaggle/working/comp_meta.json', 'w'))
print(f'COMP STACK DONE {time.time() - T0:.0f}s', flush=True)

## Spectral readout heads (`night24_helper.py`)

kirilltym's readout bank (`synthetic4404`), which the public 0.639-0.642 notebooks mixed in at 10%. It runs on CPU, on the raw features that `lora24_public` saved above. Before using the bank, the script checks that it was fit on exactly the adapter that's attached: the adapter's SHA-256 has to match between the bank, the bank's audit file and the checkpoint. In a preview run it pushes random features through every head, just to make sure they load and run with this image's scikit-learn.

In [ ]:
%%writefile night24_helper.py
"""Fork addition: Kirill Tymchishin's spectral readout heads (dataset kirilltym/ntl-spectral-style-heads-paired-4404, bank
`synthetic4404`, mixed at 10% after ATLAS by the public 0.639-0.642 notebooks) on the raw lora24_public test features that
the complementary stack saved. CPU only.  Usage: python night24_helper.py <raw.npy | random> <test.csv> <out.npz>"""
import glob, json, os, sys
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import joblib, numpy as np, pandas as pd, sklearn

bank_path = glob.glob('/kaggle/input/**/night24_readout_banks/synthetic4404.joblib', recursive=True)[0]
sys.path.insert(0, os.path.dirname(bank_path))
from readouts import predict_bank, LAYERS

raw, test_csv, out = sys.argv[1:4]
test = pd.read_csv(test_csv)
texts = test['text'].fillna('').astype(str).tolist()
audit = json.load(open(os.path.join(os.path.dirname(bank_path), 'COMPLETE.json')))['banks']['synthetic4404']
checkpoint = json.load(open(glob.glob('/kaggle/input/**/lora24/final/COMPLETE.json', recursive=True)[0]))
bank = joblib.load(bank_path)
assert bank['adapter_sha256'] == audit['adapter_sha256'] == checkpoint['files']['adapter/adapter_model.safetensors']
assert bank['layers'] == LAYERS and len(bank['labels']) == 18
if raw == 'random':   # preview: run every head of the bank on this image's scikit-learn
    z = np.random.default_rng(0).standard_normal((len(texts), 32768)).astype(np.float32)
else:
    z = np.load(raw, mmap_mode='r')
assert z.shape == (len(texts), 32768), z.shape
p, _ = predict_bank(bank, z, texts)
assert np.isfinite(p).all() and np.allclose(p.sum(1), 1, atol=1e-6)
np.savez_compressed(out, ids=test['id'].astype(str).to_numpy(dtype=str), labels=np.asarray(bank['labels'], dtype=str), probabilities=p)
print(f'spectral readout heads: {len(p)} rows, scikit-learn {sklearn.__version__}, mean top probability {p.max(1).mean():.3f}', flush=True)

## Run the stack

Runs `comp_stack.py` in its own process (so its GPU memory is gone when it finishes), then the spectral heads if their input features exist. In a commit preview it only tests the patched checkpoint search and runs the spectral bank on random input. `STACK_SECONDS` is kept because the time budget for the rest is worked out from it.

In [ ]:
import time
t0 = time.time()
os.environ['NTL_NIGHT24_RAW'] = '/tmp/night24_raw.npy'   # fork: the stack saves lora24_public's raw test features here
if not PREVIEW:
    !python comp_stack.py 2>&1 | grep -vE "Warning|warn"
    if os.path.exists('/tmp/night24_raw.npy'):
        !python night24_helper.py /tmp/night24_raw.npy {DATA}/test.csv /kaggle/working/night24_helper.npz
else:   # fork: check the patched checkpoint lookup and the spectral bank on this image
    _src = open('comp_stack.py').read()
    _ns = {'glob': glob, 'os': os, 'json': json}
    exec(_src[_src.index('def find_checkpoint('):_src.index('def load_truncated_decoder(')], _ns)
    for _mt, _h in [('qwen3', 4096), ('qwen3', 2560), ('qwen2', 3584), ('glm4', 4096)]:
        print('find_checkpoint', _mt, _h, '->', _ns['find_checkpoint'](_mt, _h), flush=True)
    print('lora24_public', glob.glob('/kaggle/input/**/lora24/final/COMPLETE.json', recursive=True), flush=True)
    !python night24_helper.py random {DATA}/test.csv /tmp/night24_preview.npz
STACK_SECONDS = time.time() - t0

## Main pipeline (`pipeline.py`)

### Arguments and data

The label order, the command-line arguments (member list, pool, mixing weights, time budget) and the data. `TINY` is the 3-row case again.

In [ ]:
%%writefile pipeline.py
"""Name That LLM inference: two-stage LoRA24 ensemble, test-prompt structure, ATLAS, class balancing."""
import os, sys, time, gc, argparse, json, re
import numpy as np, pandas as pd, torch
from scipy.optimize import linear_sum_assignment
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from transformers import AutoTokenizer, AutoModel, AutoConfig
from safetensors.torch import load_file
import gauss

LABELS = ['claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
          'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'glm-5.3', 'glm-5.3-flash',
          'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'kimi-k3', 'qwen3.8-max']
K = len(LABELS)
L2I = {l: i for i, l in enumerate(LABELS)}

p = argparse.ArgumentParser()
p.add_argument('--data', required=True)
p.add_argument('--members', required=True)
p.add_argument('--pool', required=True)
p.add_argument('--svc_w', type=float, default=1.5)
p.add_argument('--twin_w', type=float, default=1.0)
p.add_argument('--cluster_thr', type=float, default=0.15)
p.add_argument('--comp_path', default='')
p.add_argument('--comp_w', type=float, default=0.15)
p.add_argument('--atlas_dir', default='')
p.add_argument('--atlas_base', default='')
p.add_argument('--atlas_w', type=float, default=0.10)
p.add_argument('--atlas_margin', type=float, default=0.3)
p.add_argument('--atlas_bs', type=int, default=4)
p.add_argument('--atlas_chunk', type=int, default=3)
p.add_argument('--budget', type=float, default=8.3 * 3600)
p.add_argument('--helper_path', default=''); p.add_argument('--helper_w', type=float, default=0.0)   # fork: spectral readout heads
p.add_argument('--out', default='submission.parquet')
p.add_argument('--member_deadline', type=float, default=0.0)   # fork: no new LoRA adapter after this (seconds after start)
p.add_argument('--extra', required=True)
a = p.parse_args()
a.extra_w=0.5
a.tfidf_w=0.25
a.distort_w=0.10
a.perpassage=1
a.sim=''

torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
T0 = time.time()
def log(*x): print(f'[{time.time() - T0:7.0f}s]', *x, flush=True)

def log_softmax(z):
    z = z - z.max(1, keepdims=True)
    return z - np.log(np.exp(z).sum(1, keepdims=True))

train = pd.read_csv(os.path.join(a.data, 'train.csv'))
test = pd.read_csv(os.path.join(a.data, 'test.csv'))
train['text'] = train.text.fillna(''); test['text'] = test.text.fillna('')
y = train.model.map(L2I).values
TINY = len(test) < 50
log('train', len(train), 'test', len(test))




### Recovering test prompts

Content-word TF-IDF (English stop words dropped, and also any word that appears in more than 5% of passages, so only rare, topic-specific words are left), cosine similarity, then average-linkage clustering that never lets a cluster grow past 18 passages. Merging stops when the best remaining similarity is below 0.15.

On the hidden test most passages end up on their own, which is expected: it has mostly one or two passages per prompt. The group tricks further down only apply where the test itself shows a cluster.

In [ ]:
%%writefile -a pipeline.py
# ---------------------------------------------------------------- test prompt clusters
def capped_agglo(S, cap=18, thr=0.15):
    """Average-linkage clustering that never grows a cluster past `cap` (one passage per model per prompt)."""
    n = len(S); A = S.astype(np.float32).copy(); np.fill_diagonal(A, -np.inf)
    size = np.ones(n, np.int32); alive = np.ones(n, bool); lab = np.arange(n)
    while True:
        ok = (size[:, None] + size[None, :] <= cap) & alive[:, None] & alive[None, :]
        M = np.where(ok, A, -np.inf); i, j = divmod(int(np.argmax(M)), n)
        if not np.isfinite(M[i, j]) or M[i, j] < thr:
            break
        new = (size[i] * A[i] + size[j] * A[j]) / (size[i] + size[j])
        A[i] = new; A[:, i] = new; A[i, i] = -np.inf; A[j] = -np.inf; A[:, j] = -np.inf
        size[i] += size[j]; alive[j] = False; lab[lab == j] = i
    return lab

try:
    Xc = TfidfVectorizer(sublinear_tf=True, stop_words='english', min_df=2, max_df=0.05).fit_transform(test.text.values)
    groups = capped_agglo((Xc @ Xc.T).toarray(), 18, a.cluster_thr)
except ValueError:
    groups = np.arange(len(test))
groups = np.unique(groups, return_inverse=True)[1]
gsize = np.bincount(groups)[groups]
log('test clusters', groups.max() + 1, '| passages in clusters of 2-18:', int(((gsize >= 2) & (gsize <= 18)).sum()))

LP = {}




### V31 text models

Deduplicated SVC: official weight 1, external 0.5, pool 0.35. TF-IDF and distortion use official plus matched-clean examples.


In [ ]:
%%writefile -a pipeline.py
import scipy.sparse as sp
EXT_T, EXT_Y = np.array([], dtype=object), np.array([], dtype=int)
EXTRA_DF = pd.DataFrame(columns=['text', 'model'])
def _norm_text_key(value):
    import unicodedata
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', str(value or ''))).strip()
if a.extra:
    _E = pd.read_parquet(a.extra) if a.extra.endswith('.parquet') else pd.read_csv(a.extra)
    if not {'text', 'model'}.issubset(_E.columns):
        raise ValueError(f'External file must contain text/model columns: {a.extra}')
    _E = _E[_E.model.isin(LABELS)].copy()
    _E['text'] = _E.text.fillna('').astype(str)
    # In --sim mode, remove any external exact-text matches to held-out official rows.
    # For final submission, retain the public external pool as provided by the source.
    if a.sim:
        _held_keys = {_norm_text_key(t) for t in test.text.values}
        _E = _E[~_E.text.map(_norm_text_key).isin(_held_keys)].copy()
    _official_keys = {_norm_text_key(t) for t in train.text.values}
    _seen = set(_official_keys)
    _keep = []
    for _t in _E.text.values:
        _k = _norm_text_key(_t)
        _ok = bool(_k) and _k not in _seen
        _keep.append(_ok)
        if _ok: _seen.add(_k)
    EXTRA_DF = _E.loc[_keep, ['text', 'model']].reset_index(drop=True)
    EXT_T = EXTRA_DF.text.astype(str).values
    EXT_Y = EXTRA_DF.model.map(L2I).values.astype(int)
    log('matched-clean extra rows after exact de-dup', len(EXT_T))
FIT_T = np.concatenate([train.text.values, EXT_T])
FIT_Y = np.concatenate([y, EXT_Y])
FIT_W = np.concatenate([np.ones(len(train), dtype=np.float32), np.full(len(EXT_T), a.extra_w, dtype=np.float32)])
# ---------------- 2. TF-IDF centred LR
if a.tfidf_w > 0:
    from sklearn.linear_model import LogisticRegression
    mats_tr, mats_te = [], []
    for kw in [dict(analyzer='char_wb', ngram_range=(2, 5), min_df=3, max_features=60000),
               dict(token_pattern=r"\w+|[^\w\s]", ngram_range=(1, 2), min_df=3, max_features=30000)]:
        vec = TfidfVectorizer(sublinear_tf=True, lowercase=False, **kw).fit(FIT_T if a.perpassage else train.text.values)
        mats_tr.append(vec.transform(FIT_T if a.perpassage else train.text.values)); mats_te.append(vec.transform(test.text.values))
    if a.perpassage:
        LP['tfidf'] = LogisticRegression(C=20, max_iter=3000).fit(sp.hstack(mats_tr).tocsr(), FIT_Y, sample_weight=FIT_W).predict_log_proba(sp.hstack(mats_te).tocsr()) * a.tfidf_w
    else:
        Xtr = center_groups(sp.hstack(mats_tr).toarray().astype(np.float32), Gtr)
        Xte = center_groups(sp.hstack(mats_te).toarray().astype(np.float32), Gte)
        LP['tfidf'] = LogisticRegression(C=5, max_iter=3000).fit(Xtr, y).predict_log_proba(Xte) * a.tfidf_w
        del Xtr, Xte
    gc.collect(); log('tfidf done')
else:
    LP['tfidf'] = np.zeros((len(test), 18), dtype=np.float32)
    log('tfidf skipped (zero ensemble weight)')
if a.svc_w > 0:  # four-view SVC: official + matched-clean + pool, with source weights and exact de-duplication
    import re as _re
    from scipy.sparse import hstack as _hs
    from sklearn.svm import LinearSVC
    _FW = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
    _shape = lambda t: _re.sub(r"[^\W\d_]+", lambda m_: "X" if m_[0].isupper() else "x", t)
    _func = lambda t: " ".join(w.lower() if w.lower() in _FW or not w.isalnum() else "CONTENT" for w in _re.findall(r"\w+|[^\w\s]", t))
    _parts = []
    _official = train[['text', 'model']].copy(); _official['text'] = _official.text.fillna('').astype(str)
    _official['_srcw'] = 1.0
    _parts.append(_official)
    _seen = {_norm_text_key(t) for t in _official.text.values}
    if len(EXTRA_DF):
        _e = EXTRA_DF[['text', 'model']].copy(); _e['_srcw'] = float(a.extra_w)
        _keep = []
        for _t in _e.text.values:
            _k = _norm_text_key(_t); _ok = bool(_k) and _k not in _seen
            _keep.append(_ok)
            if _ok: _seen.add(_k)
        _e = _e.loc[_keep].copy(); _parts.append(_e)
    if a.pool and os.path.exists(a.pool):
        _P = pd.read_parquet(a.pool)
        if not {'text', 'model'}.issubset(_P.columns):
            raise ValueError(f'Pool must contain text/model columns: {a.pool}')
        _P = _P[_P.model.isin(LABELS)][['text', 'model']].copy()
        _P['text'] = _P.text.fillna('').astype(str); _P['_srcw'] = 0.35
        if a.sim:
            _held_keys = {_norm_text_key(t) for t in test.text.values}
            _P = _P[~_P.text.map(_norm_text_key).isin(_held_keys)].copy()
        _keep = []
        for _t in _P.text.values:
            _k = _norm_text_key(_t); _ok = bool(_k) and _k not in _seen
            _keep.append(_ok)
            if _ok: _seen.add(_k)
        _P = _P.loc[_keep].copy(); _parts.append(_P)
    _fitdf = pd.concat(_parts, ignore_index=True)
    _fit = _fitdf.text.astype(str).tolist(); _fy = _fitdf.model.map(L2I).to_numpy(dtype=int); _sw = _fitdf['_srcw'].to_numpy(dtype=np.float32)
    log(f'svc4 fit rows={len(_fit)} source_weighted; external clean={len(EXTRA_DF)}')
    _A, _B = [], []
    for view, lo, hi, mf, wv in [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("function", 1, 3, 50000, 0.5)]:
        prep = _shape if view == "shape" else _func if view == "function" else (lambda s_: s_)
        an = "char" if view == "shape" else "word" if view == "function" else view
        o = dict(analyzer=an, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2, max_features=mf, dtype=np.float32)
        if an == "word": o["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
        _v = TfidfVectorizer(**o); _A.append(_v.fit_transform([prep(t) for t in _fit]) * wv); _B.append(_v.transform([prep(t) for t in test.text.values]) * wv)
    _svc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight='balanced')
    _svc.fit(_hs(_A, format='csr'), _fy, sample_weight=_sw)
    _d = _svc.decision_function(_hs(_B, format='csr'))
    LP['svc4'] = (_d - _d.max(1, keepdims=True) - np.log(np.exp(_d - _d.max(1, keepdims=True)).sum(1, keepdims=True))) * a.svc_w
    del _A, _B, _parts, _fitdf, _svc; gc.collect(); log('svc4 done')
if a.distort_w > 0:  # Stamatatos-style text distortion: keep top-100 frequent words + punctuation, mask the rest
    import re, collections
    wre = re.compile(r"[A-Za-z’']+")
    cnt = collections.Counter(w.lower() for t in FIT_T for w in wre.findall(t))
    keep = {w for w, _ in cnt.most_common(100)}
    dist = lambda arr: [wre.sub(lambda m_: m_.group(0) if m_.group(0).lower() in keep else '*', t) for t in arr]
    Dtr, Dte = dist(FIT_T), dist(test.text.values)
    vs = [TfidfVectorizer(sublinear_tf=True, lowercase=False, analyzer='char', ngram_range=(2, 6), min_df=2),
          TfidfVectorizer(sublinear_tf=True, lowercase=True, token_pattern=r"[^\s]+", ngram_range=(1, 3), min_df=2)]
    Xa = sp.hstack([v_.fit_transform(Dtr) for v_ in vs]).tocsr(); Xb = sp.hstack([v_.transform(Dte) for v_ in vs]).tocsr()
    LP['distort'] = LogisticRegression(C=10, max_iter=3000).fit(Xa, FIT_Y, sample_weight=FIT_W).predict_log_proba(Xb) * a.distort_w
    log('distort done')



### Two-stage LoRA members

`lora24` loads a base model cut to its first `keep` layers, once. Then for every adapter it:

1. merges the LoRA update into the weights (`W += B·A·alpha/r`),
2. runs the passages and mean-pools several layers (the first token is left out), each through its own LayerNorm,
3. applies the plain head, and the centred head on recovered groups,
4. subtracts the update again, so the next adapter starts from the clean base.

Merging is faster than running PEFT hooks, and loading the base only once is what makes five fold adapters plus a full-data adapter affordable for a 14B model. The predictions are averaged over the adapters.

Details that matter:

- Heads trained with a different label order are reordered from their `meta.json`.
- Checkpoints trained with an end token appended (Nishant's) get the same token id here. Kaggle's Qwen3-14B-Base tokenizer reports a different end token, and according to dhruv using it changes 14% of that member's predictions.
- If fp16 overflows on a batch, that batch is redone in bf16.
- The time guard (mine): after the deadline no new adapter is started, and the average is taken over the adapters that did run.

In [ ]:
%%writefile -a pipeline.py
# ---------------------------------------------------------------- two-stage LoRA24 members
def batches(enc, bs):
    order = np.argsort([len(x) for x in enc])
    for i in range(0, len(order), bs):
        b = order[i:i + bs]; L = max(len(enc[j]) for j in b)
        ids = torch.zeros((len(b), L), dtype=torch.long); att = torch.zeros((len(b), L), dtype=torch.long)
        for r, j in enumerate(b):
            ids[r, :len(enc[j])] = torch.tensor(enc[j]); att[r, :len(enc[j])] = 1
        yield b, ids, att

@torch.no_grad()
def lora24(spec, texts, groups=None, train_texts=None):
    """Truncated base loaded once; each adapter is merged into the weights (W += BA * alpha/r), read, then unmerged.
    Features: masked mean-pool (first token dropped) of several layers, each through its own LayerNorm.
    Returns plain-head and centred-head log-probs averaged over adapters, plus (test, train) features of the last adapter."""
    tok = AutoTokenizer.from_pretrained(spec.get('tokenizer_path', spec['path']))
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    cfg = AutoConfig.from_pretrained(spec['path']); tc = getattr(cfg, 'text_config', cfg)
    tc.num_hidden_layers = spec['keep']; tc.use_cache = False
    if getattr(tc, 'layer_types', None) is not None:
        tc.layer_types = tc.layer_types[:spec['keep']]
    m = AutoModel.from_pretrained(spec['path'], config=cfg, dtype=torch.float16, device_map='auto').eval()
    if spec.get('no_final_norm'):
        m.norm = torch.nn.Identity()
    mods = dict(m.named_modules()); dev = next(m.parameters()).device; layers = spec['layers']
    eos = spec.get('eos_id')   # checkpoints trained with an end token appended need the same id here
    def tokenize(tx):
        ids = tok(list(tx), truncation=True, max_length=spec['max_len'] - (1 if eos is not None else 0))['input_ids']
        return [x + [eos] for x in ids] if eos is not None else ids
    if spec.get('our_poolv5'):
        def tokenize(tx):
            return [x + [eos] for x in tok(list(tx), add_special_tokens=False, truncation=True, max_length=447)['input_ids']]
    enc = tokenize(texts)
    enc_tr = tokenize(train_texts) if train_texts is not None else None

    def featurize(enc_, norms, hid):
        Z = np.zeros((len(enc_), hid * len(layers)), np.float32)
        for b, ids, att in batches(enc_, spec['bs']):
            ids[att == 0] = pad
            hs = m(input_ids=ids.to(dev), attention_mask=att.to(dev), output_hidden_states=True, use_cache=False).hidden_states
            if not all(torch.isfinite(hs[l]).all() for l in layers):   # fp16 overflow: redo the batch in bf16
                m.to(torch.bfloat16); hs = m(input_ids=ids.to(dev), attention_mask=att.to(dev), output_hidden_states=True, use_cache=False).hidden_states; m.to(torch.float16)
            mask = att.unsqueeze(-1).float(); mask[:, 0] = 0
            Z[b] = torch.cat([nm((hs[l].float().cpu() * mask).sum(1) / mask.sum(1)) for l, nm in zip(layers, norms)], -1).numpy()
        return Z

    plain, cent, feats = 0.0, 0.0, None
    n_used = 0
    for ai, ad in enumerate(spec['adapters']):
        if ai > 0 and a.member_deadline > 0 and time.time() - T0 > a.member_deadline:   # fork: time guard
            log(f'  {spec["name"]}: time guard, {ai}/{len(spec["adapters"])} adapters used'); break
        n_used += 1
        adp = os.path.join(ad, 'adapter') if os.path.exists(os.path.join(ad, 'adapter', 'adapter_config.json')) else ad
        acfg = json.load(open(os.path.join(adp, 'adapter_config.json'))); scale = acfg['lora_alpha'] / acfg['r']
        sd = load_file(os.path.join(adp, 'adapter_model.safetensors')); merged = []
        for k, A in sd.items():
            if '.lora_A.' not in k:
                continue
            name = k.replace('base_model.model.', '', 1).split('.lora_A.')[0]
            if name not in mods and 'model.' + name in mods:
                name = 'model.' + name
            W = mods[name].weight; B = sd[k.replace('.lora_A.', '.lora_B.')]
            W += ((B.to(W.device, torch.float32) @ A.to(W.device, torch.float32)) * scale).to(W.dtype)
            merged.append((W, B, A))
        H = torch.load(os.path.join(ad, 'heads.pt'), map_location='cpu') if os.path.exists(os.path.join(ad, 'heads.pt')) \
            else dict(load_file(os.path.join(ad, 'heads.safetensors')))
        if os.path.exists(os.path.join(ad, 'meta.json')):
            order = [json.load(open(os.path.join(ad, 'meta.json')))['labels'].index(l) for l in LABELS]
            for h in ('plain', 'centered'):
                H[f'{h}.weight'], H[f'{h}.bias'] = H[f'{h}.weight'][order], H[f'{h}.bias'][order]
        hid = H['norms.0.weight'].shape[0]
        norms = [torch.nn.LayerNorm(hid) for _ in layers]
        for i, nm in enumerate(norms):
            nm.load_state_dict({'weight': H[f'norms.{i}.weight'], 'bias': H[f'norms.{i}.bias']})
        Z = featurize(enc, norms, hid)
        if enc_tr is not None and ai == len(spec['adapters']) - 1:
            feats = (Z.copy(), featurize(enc_tr, norms, hid))
        for W, B, A in merged:
            W -= ((B.to(W.device, torch.float32) @ A.to(W.device, torch.float32)) * scale).to(W.dtype)
        del merged, sd; gc.collect(); torch.cuda.empty_cache()
        head = lambda h, X: log_softmax(X @ H[f'{h}.weight'].float().numpy().T + H[f'{h}.bias'].float().numpy())
        plain = plain + head('plain', Z) / len(spec['adapters'])
        if groups is not None:
            Zc = Z.copy()
            for g in np.unique(groups):
                Zc[groups == g] -= Zc[groups == g].mean(0)
            cent = cent + head('centered', Zc) / len(spec['adapters'])
        log(f'  {spec["name"]}: {os.path.basename(ad)}')
    del m, mods; gc.collect(); torch.cuda.empty_cache()
    if 0 < n_used < len(spec['adapters']):   # fork: average over the adapters actually used
        plain = plain * len(spec['adapters']) / n_used
        if groups is not None:
            cent = cent * len(spec['adapters']) / n_used
    return plain, (cent if groups is not None else None), feats



### DeBERTa, and running the members

`encoder_member` is Nishant's fine-tuned DeBERTa-v3-large: fp32, mean of the last layer, linear head.

The loop runs each member and stores up to three terms for it:

- `name`: the plain head × the member's weight,
- `name_twin`: the centred head on passages in recovered groups × `twin`,
- `name_gauss`: the shared-prompt Gaussian on the full-data adapter's features, for the two members that have a `gauss` weight, on groups of 2 to 18 passages, at temperature 3.

In a 3-row run every LoRA member first has to classify 40 training passages it was trained on with more than 50% accuracy. If a checkpoint is being read wrongly, this is where it shows.

In [ ]:
%%writefile -a pipeline.py
@torch.no_grad()
def encoder_member(ckpt, texts):
    """Fully fine-tuned encoder (DeBERTa / ModernBERT): fp32 weights, mean of the last layer, linear head."""
    meta = json.load(open(os.path.join(ckpt, 'meta.json'))); perm = [meta['labels'].index(l) for l in LABELS]
    head = load_file(os.path.join(ckpt, 'head.safetensors')); W = head['weight'].float().numpy()[perm]; b = head['bias'].float().numpy()[perm]
    etok = AutoTokenizer.from_pretrained(os.path.join(ckpt, 'encoder'))
    enc = AutoModel.from_pretrained(os.path.join(ckpt, 'encoder'), dtype=torch.float32).to('cuda:0').eval()
    def feats(tx):
        ids = etok(list(tx), truncation=True, max_length=meta['max_len'])['input_ids']
        order = np.argsort([len(x) for x in ids], kind='stable'); Z = np.zeros((len(ids), enc.config.hidden_size), np.float32)
        for s in range(0, len(order), 32):
            idx = order[s:s + 32]; bt = etok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to('cuda:0')
            mk = bt['attention_mask'].unsqueeze(-1).float()
            Z[idx] = ((enc(**bt).last_hidden_state * mk).sum(1) / mk.sum(1).clamp(min=1)).cpu().numpy()
        return Z
    if TINY:
        acc = float(((feats(train.text.values[:40]) @ W.T + b).argmax(1) == y[:40]).mean()); log(f'encoder in-sample check {acc:.2f}'); assert acc > 0.5
    out = log_softmax(feats(texts) @ W.T + b)
    del enc; gc.collect(); torch.cuda.empty_cache()
    return out

GAUSS_CFG = dict(red='wpca', d=128, se_shrink='lw', sp_shrink='lw', lam_floor=1e-4, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0)
grouped = (gsize >= 2) & (gsize <= 18)
for spec in json.loads(a.members):
    name, w = spec['name'], spec['w']
    if a.member_deadline > 0 and time.time() - T0 > a.member_deadline and 'qwen3_14b' in LP:   # fork: time guard
        log(name, 'skipped: time guard'); continue
    if spec.get('type') == 'enc':
        LP[name] = encoder_member(spec['ckpt'], test.text.values) * w; log(name, 'done'); continue
    if TINY:   # interactive 3-row run: the heads must reproduce their fit on training rows
        acc = float((lora24(spec, train.text.values[:40])[0].argmax(1) == y[:40]).mean())
        log(f'{name} in-sample check {acc:.2f}'); assert acc > 0.5, name
    use_gauss = spec.get('gauss', 0) > 0
    plain, cent, feats = lora24(spec, test.text.values, groups, train.text.values if use_gauss else None)
    LP[name] = plain * w
    LP[name + '_twin'] = np.where((gsize >= 2)[:, None], cent, 0.0) * a.twin_w * w * spec['twin']
    if use_gauss and feats is not None:   # shared-prompt Gaussian group head on the full-data adapter's features (fork: none after a time-guard cut)
        Zte, Ztr = feats
        prm = gauss.fit(Ztr, y, train.prompt_id.values, GAUSS_CFG); prm['T1'] = prm['Tg'] = 3.0
        assert np.allclose(np.exp(gauss.scores(Ztr[[0, 1]], np.zeros(2, dtype=int), prm)).sum(1), 1)
        G = np.zeros((len(test), K))
        if grouped.any():
            G[grouped] = gauss.scores(Zte[grouped], np.unique(groups[grouped], return_inverse=True)[1], prm)
        LP[name + '_gauss'] = G * spec['gauss']
    gc.collect(); torch.cuda.empty_cache(); log(name, 'done')




### Blend

All terms are added and divided by the number of non-zero terms, then turned into probabilities. The complementary stack comes in here at 15%, mixed in probability space, after a check that its row order matches the test file.

In [ ]:
%%writefile -a pipeline.py
# ---------------------------------------------------------------- blend
n_terms = sum(np.abs(v).max() > 0 for v in LP.values())
z = sum(LP.values()) / max(1, n_terms)
P = np.exp(z - z.max(1, keepdims=True)); P /= P.sum(1, keepdims=True)

if a.comp_w > 0 and a.comp_path and os.path.exists(a.comp_path):   # complementary frozen-feature stack, run in its own process
    meta = json.load(open(a.comp_path.replace('comp_logit.npy', 'comp_meta.json')))
    assert meta['ids'] == test['id'].astype(str).tolist()
    cl = np.load(a.comp_path)[:, [meta['labels'].index(l) for l in LABELS]]
    C = np.exp(cl - cl.max(1, keepdims=True)); C /= C.sum(1, keepdims=True)
    P = (1 - a.comp_w) * P + a.comp_w * C; P /= P.sum(1, keepdims=True)
    log(f"complementary stack mixed at {a.comp_w}")
else:
    log('complementary stack not available, skipped')




### ATLAS20

20% on rows with top-two margin below 0.3; shortest first until the existing deadline.


In [ ]:
%%writefile -a pipeline.py
@torch.no_grad()
def atlas(bundle, rows):
    """Source-conditioned likelihood (ATLAS, arXiv 2610.19583) from a public bundle; shortest passages first,
    stops early if the token-proportional projection would overrun the time budget."""
    sys.path.insert(0, bundle)
    from ntl_atlas.model import load_model
    from ntl_rlcd.data import LABELS as ALAB
    cfg = json.load(open(os.path.join(bundle, 'atlas_bundle.json')))['training_config']
    model, tok = load_model(a.atlas_base, ALAB, cfg, device='auto')
    try:
        model.load_trainable(load_file(os.path.join(bundle, 'adapters.safetensors'))); model.eval()
        enc = tok(test.text.values[rows].tolist(), truncation=True, max_length=cfg['max_length'])['input_ids']
        order = np.argsort([len(t) for t in enc], kind='stable'); out = np.zeros((len(rows), K), np.float32)
        ctok = np.cumsum([len(enc[i]) for i in order]).astype(np.float64); t1 = time.time()
        scored = np.zeros(len(rows), bool)   # fork: keep the passages scored within the budget instead of dropping ATLAS
        for s in range(0, len(order), a.atlas_bs):
            if time.time() - T0 > a.budget:
                log(f'  atlas time budget reached after {int(scored.sum())}/{len(order)} passages'); break
            idx = order[s:s + a.atlas_bs]
            pack = tok.pad({'input_ids': [enc[i] for i in idx]}, padding=True, return_tensors='pt').to(model.device)
            out[idx] = model.score(pack['input_ids'], pack['attention_mask'], source_chunk=a.atlas_chunk)[0].float().cpu().numpy()
            scored[idx] = True
            done = s + len(idx)
            if s % (25 * a.atlas_bs) == 0:
                eta = (time.time() - T0) + (time.time() - t1) / ctok[done - 1] * (ctok[-1] - ctok[done - 1]) * 1.1
                log(f'  atlas {done}/{len(order)}, projected finish {eta:.0f}s')
        assert np.isfinite(out[scored]).all() and np.allclose(out[scored].sum(1), 1, atol=1e-4)
        return out[:, [ALAB.index(l) for l in LABELS]], scored
    finally:
        del model; gc.collect(); torch.cuda.empty_cache()

if a.atlas_dir and a.atlas_w > 0:   # a 10% mix can only flip rows where the ensemble is close, so score only those
    top2 = np.sort(P, 1)[:, -2:]
    rows = np.arange(len(test)) if TINY else np.where(top2[:, 1] - top2[:, 0] < a.atlas_margin)[0]
    log(f'atlas on {len(rows)}/{len(test)} passages')
    try:
        if time.time() - T0 > a.budget - 300:   # fork: no time left to load the ATLAS model
            raise RuntimeError('time budget')
        _A, _ok = atlas(a.atlas_dir, rows)
        P[rows[_ok]] = (1 - a.atlas_w) * P[rows[_ok]] + a.atlas_w * _A[_ok]
        P /= P.sum(1, keepdims=True); log(f'atlas mixed at {a.atlas_w} on {int(_ok.sum())}/{len(rows)} passages')
    except Exception as e:
        log('atlas skipped:', repr(e))




### Spectral readout heads

Mixed in at 10%, after ATLAS and before balancing. If the file is missing or its ids don't match the test file, it's skipped with a log line rather than an error.

In [ ]:
%%writefile -a pipeline.py
if a.helper_w > 0 and a.helper_path and os.path.exists(a.helper_path):   # fork: Kirill's spectral readout heads, after ATLAS, before balancing
    try:
        with np.load(a.helper_path) as hz:
            _pos = {i: r for r, i in enumerate(hz['ids'].astype(str))}; _hl = hz['labels'].astype(str).tolist()
            H = hz['probabilities'][[_pos[i] for i in test['id'].astype(str)]][:, [_hl.index(l) for l in LABELS]]
        assert H.shape == P.shape and np.isfinite(H).all() and np.allclose(H.sum(1), 1, atol=1e-4)
        P = (1 - a.helper_w) * P + a.helper_w * H; P /= P.sum(1, keepdims=True); log(f'spectral readout heads mixed at {a.helper_w}')
    except Exception as e:
        log('spectral readout heads skipped:', repr(e))
else:
    log('spectral readout heads not available, skipped')




### Soft balancing and typography

V31 soft Sinkhorn then its Bayesian 37-marker correction with confidence, length and coverage gates. Replaces hard marker bans and dense-group assignment.


In [ ]:
%%writefile -a pipeline.py
import scipy.special as _v38sp
if len(test) >= 18:
    prior = np.bincount(y, minlength=K).astype(np.float64) + 1
    prior /= prior.sum()
    pred = np.clip(P.mean(0), 1e-6, None); pred /= pred.sum()
    target = 0.65 * pred + 0.35 * prior
    Q = P.copy()
    for _ in range(100):
        Q *= (len(Q) * target[None,:]) / np.clip(Q.sum(0,keepdims=True),1e-12,None)
        Q /= np.clip(Q.sum(1,keepdims=True),1e-12,None)
    score = 0.55*np.log(P+1e-12) + 0.45*np.log(Q+1e-12)
else:
    score = np.log(P+1e-12)
_marks = [' — ', '—', '–', '’', '‘', '“', '”', '«', '»', '"', ';', ':', '(', ')', '[', ']', '**', '`', '…', '•', '\n', '\n\n', 'e.g.', 'i.e.', 'However,', 'Overall,', 'In conclusion', 'Therefore,', 'Moreover,', '\t', ' - ', ' = ', '\n- ', '\n1.', '###', '```', '> ']
_Xtr = np.array([[float(_m in _t) for _m in _marks] for _t in train.text.values], dtype=np.float32)
_Xte = np.array([[float(_m in _t) for _m in _marks] for _t in test.text.values], dtype=np.float32)
_freq = np.bincount(y, minlength=len(LABELS)).astype(np.float64)
_hits = np.stack([_Xtr[y == _k].sum(axis=0) for _k in range(len(LABELS))], axis=0).astype(np.float64)
# Hierarchical Beta-Bernoulli smoothing: class-specific marker rates shrink toward the global rate.
_global_rate = (_Xtr.sum(axis=0) + 2.0) / (len(_Xtr) + 4.0)
_kappa = 24.0
_rate = (_hits + _kappa * _global_rate[None, :]) / (_freq[:, None] + _kappa)
_rate = np.clip(_rate, 0.01, 0.99)
_global_rate = np.clip(_global_rate, 0.01, 0.99)
_log_rate = np.log(_rate); _log_not_rate = np.log1p(-_rate)
_base_log_rate = np.log(_global_rate); _base_log_not_rate = np.log1p(-_global_rate)
# Weight markers by how consistently they separate classes, capped to avoid one marker dominating.
_signal = np.sqrt(np.mean((_rate - _global_rate[None, :]) ** 2, axis=0))
_fw = _signal / max(float(np.median(_signal[_signal > 0])) if np.any(_signal > 0) else 1.0, 1e-6)
_fw = np.clip(_fw, 0.25, 2.0)
# Matrix form: weighted Bernoulli log likelihood ratio for each class.
_evidence = (_Xte * _fw[None, :]) @ _log_rate.T + ((1.0 - _Xte) * _fw[None, :]) @ _log_not_rate.T
_evidence -= (((_Xte * _fw[None, :]) @ _base_log_rate) + (((1.0 - _Xte) * _fw[None, :]) @ _base_log_not_rate))[:, None]
_evidence -= _evidence.mean(axis=1, keepdims=True)
_evidence = np.clip(1.5 * _evidence / max(float(_fw.sum()), 1e-6), -0.30, 0.30)
_raw = score.copy()
_probs = _v38sp.softmax(_raw, axis=1)
_sorted = np.sort(_probs, axis=1)
_margin = _sorted[:, -1] - _sorted[:, -2]
_confidence_gate = np.clip((0.60 - _margin) / 0.60, 0.0, 1.0)
_word_count = np.array([len(re.findall(r'\b\w+\b', str(_t))) for _t in test.text.values], dtype=np.float32)
_length_gate = np.clip(np.sqrt(np.maximum(_word_count, 1.0) / 80.0), 0.40, 1.0)
_active_markers = _Xte.sum(axis=1)
_coverage_gate = np.clip(np.sqrt((_active_markers + 1.0) / 4.0), 0.45, 1.0)
_gate = _confidence_gate * _length_gate * _coverage_gate
_safe_logits = _raw + _gate[:, None] * _evidence

sub = pd.DataFrame({'id':test.id.values, 'model':np.array(LABELS)[_safe_logits.argmax(1)]})
assert sub.id.is_unique and len(sub)==len(test) and sub.model.isin(LABELS).all()
sub.to_parquet(a.out,index=False)
log('Typography changed',int((score.argmax(1)!=_safe_logits.argmax(1)).sum()),'rows; wrote',a.out)


## Members

The original Llama-free roster and weights, with our Pool v5 in the Qwen3-8B slot by default. Disable `USE_OUR_POOLV5` for the public fold-ensemble control.


In [ ]:
L = [2, 4, 6, 8, 12, 16, 20, 24]
MEMBERS = [
    dict(name='qwen3_14b', path=QWEN14, adapters=folds('q14v5'), keep=27, layers=L, w=3.0, bs=8,  max_len=384, twin=0.35, gauss=0.70),
    dict(name='qwen3_8b',  path=QWEN8,  adapters=([str(OUR_POOLV5)] if USE_OUR_POOLV5 else folds('q8v5')),  keep=24, layers=L, w=2.0, bs=16, max_len=384, twin=0.35, gauss=0.44),
    dict(name='nishant_14b',   path=QWEN14,    adapters=[N14], keep=N14_CFG['keep'], layers=N14_CFG['layers'], w=2.5, bs=8,
         max_len=N14_CFG['max_len'], twin=0.35, no_final_norm=True, eos_id=N14_CFG['eos_id']),
    dict(name='qwen25_14b', path=QWEN25_14, adapters=folds('q25_14v5'), keep=32, layers=[3, 5, 8, 11, 16, 21, 27, 32], w=2.0, bs=8, max_len=384, twin=0.35),
    dict(name='deberta',    type='enc', ckpt=DEBERTA,    w=1.0),
]
BUDGET = int(8.3 * 3600 - STACK_SECONDS - 600)
MEMBER_DEADLINE = BUDGET - 900   # fork: no new LoRA adapter after this (pipeline seconds); ATLAS keeps what it scores by BUDGET

if USE_OUR_POOLV5:
    MEMBERS[1].update(name='our_poolv5',gauss=0.0,max_len=448,bs=8,tokenizer_path=OUR_TOKENIZER,eos_id=OUR_EOS,our_poolv5=True,no_final_norm=False)
print('Qwen3-8B member:',MEMBERS[1])


## Run

ATLAS20, complementary stack 15%, spectral heads 10%. The source three-row preview is retained; hidden-test execution runs the full pipeline.


In [ ]:
import subprocess, sys
if not PREVIEW:
    subprocess.run([sys.executable,'pipeline.py','--data',DATA,'--members',json.dumps(MEMBERS),
        '--pool',POOL,'--extra',EXTRA,'--comp_path','/kaggle/working/comp_logit.npy','--comp_w','0.15',
        '--atlas_dir',ATLAS,'--atlas_base',QWEN8,'--atlas_w','0.20','--atlas_margin','0.3',
        '--budget',str(BUDGET),'--member_deadline',str(MEMBER_DEADLINE),
        '--helper_path','/kaggle/working/night24_helper.npz','--helper_w','0.10'],check=True)


The first rows of the submission. In a commit run these are just the placeholder rows.

In [ ]:
import pandas as pd
pd.read_parquet('submission.parquet').head()

## What didn't help (for me)

- **lightningv08's Qwen3.8-27B classifier at 10%.** +0.004 on the older base (0.643 to 0.647), but on this version it went the other way: 0.646 with it, 0.649 without. The 3-bit GSQ build of the same model scored 0.647. I left it out.
- **Re-weighting strong public blends.** Every weight change I tried on a 0.64 public blend scored lower, by 0.002 to 0.007.
- **Heavier test-time prompt tricks.** Strong cluster centring and Hungarian assignment on small groups hurt for me early on, because the test is mostly singletons. Light centred twins, and assignment only in near-complete groups, is what survives.
- From dhruv's notes: a fully fine-tuned ModernBERT-large member was +0.6 OOF but -0.6 on the leaderboard.

## Notes

- **Validate per passage, grouped by prompt** (`GroupKFold` on `prompt_id`). Random splits put the same prompt on both sides and look much better than they are.
- **Most of the score is data.** From dhruv's write-up: four different backbones trained on a 38k pool all finished near 0.57, and the same recipe on 120k passages gave 0.634 for the 14B.
- **End tokens and label orders** are the classic way to load a checkpoint "successfully" and still lose points. Read each checkpoint's own config.
- If you fork this, the commit run takes about a minute. The real run happens when you submit.

## Credits

Almost everything here is other people's public work:

- [dhruvtibarewal](https://www.kaggle.com/dhruvtibarewal): the [Fully Reproducible](https://www.kaggle.com/code/dhruvtibarewal/name-that-llm-fully-reproducible) pipeline this is built on, the 120k pool (`nlm-matched-clean`) and the Qwen3 / Qwen2.5 LoRA fold adapters (`nlm-l24-adapters-v5`, `nlm-q25-v5`).
- [nishantdahal](https://www.kaggle.com/nishantdahal): the two-stage LoRA recipe ([part two](https://www.kaggle.com/code/nishantdahal/two-stage-lora-part-two)) and the part-two Qwen3-14B and DeBERTa-v3-large checkpoints (`ntl-handwriting-blend`).
- [prvsiyan](https://www.kaggle.com/prvsiyan): the shared-prompt Gaussian, the complementary stack, and the extended passage dataset used as in-style data.
- [kirilltym](https://www.kaggle.com/kirilltym): the combined ensemble and the spectral readout heads (`ntl-spectral-style-heads-paired-4404`).
- [josephadamski91](https://www.kaggle.com/josephadamski91): ATLAS (`ntl-atlas-qwen3-8b-v1`, CC BY-SA 4.0).
- [lightningv08](https://www.kaggle.com/lightningv08): the inference assets that bring `lora24_public` back.
- [lucasboesen](https://www.kaggle.com/lucasboesen): the GLM-4-9B, encoder, ELECTRA, desklib, LoRA24 seed-7 and token-attention notebooks behind several stack members, and a free-proxy generation dataset.
- [vnton37nguyn](https://www.kaggle.com/vnton37nguyn): the in-style Qwen2.5-7B LoRA.
- [zbigniewkirchner](https://www.kaggle.com/zbigniewkirchner): the Qwen3-8B LoRA with a Claude head.
- [pjmathematician](https://www.kaggle.com/pjmathematician): UltraBERT-380M.
- [nuridudak](https://www.kaggle.com/nuridudak): the 122k clean pool that the 120k pool starts from.
- sangam21, for an earlier port of the Gaussian head.
- The authors of the generation datasets used as in-style passages (seyitkaangunes, abdullah1afify, dophuchung, lucasboesen, prvsiyan) and everyone whose passages ended up in the public pools.
- The Qwen team for Qwen3 (14B, 8B, 4B) and Qwen2.5 (14B, 7B), and the authors of the other frozen models in the stack.

If this was useful, please upvote the work it stands on, dhruv's and Nishant's notebooks first.